# Graph walks and attention-head ablations

*Author: Federico Viola · November 2026*

*Experiment 1 of [In-context graph learning in language models: induction circuits, learning curves and scaling laws](https://github.com/federicov99/in-context-graph-learning): the repository holds the write-up, the results and both notebooks.*

This notebook repeats and extends the head-ablation experiment of **Arditi (2026)**
([ICLR 2026 blog post](https://iclr-blogposts.github.io/2026/blog/2026/iclr-induction/)) on the in-context
graph-tracing task of **Park et al. (2025), [*ICLR: In-Context Learning of
Representations*](https://arxiv.org/abs/2501.00070)**.

**The task.** Every vertex of a graph gets a single-token word, and the model reads a random walk over the graph,
written as words separated by spaces. After each word we measure the **valid-neighbour probability**: the total
probability the model gives to the neighbours of the current word (Park et al.'s rule-following accuracy, the measure
Arditi also uses). The model's weights never change.

**The ablations.** Attention heads are ranked by how strongly they act as *induction heads* (attending to the word that
followed an earlier occurrence of the current word) or as *previous-token heads* (attending to the word just before),
using Arditi's synthetic probe. The top k heads of each kind, and the union of both, are then switched off by setting
their outputs to zero, in three ways:
- **everywhere**: at every position;
- **final word only**: only at the position that makes the prediction, with the history computed normally;
- **history only**: at every earlier position, but not at the position that makes the prediction.

Random heads, and optionally heads drawn from the same layers as each group, serve as controls.

The scopes separate two roles. In the textbook induction circuit, previous-token heads write "the word before me" into
each position of the history, and induction heads at the current position use that information to find and copy the
word that followed the current word before. The circuit therefore predicts that previous-token heads matter mostly in
the **history** and induction heads mostly at the **final word**.

**What you get**
1. Valid-neighbour probability against context length for every head group, head count and scope, with ±1 standard
   deviation across walks.
2. A summary table, as CSV and as an image: for each condition, the drop in valid-neighbour probability relative to
   the intact model over the second half of the walk, with 95% bootstrap intervals over walks, the drop of the matching
   control, and a *crossover* statistic that tests the prediction above.
3. PCA maps of the words' mean hidden states in every condition, and their alignment with the graph.
4. All the numbers behind the figures, with a record of the exact model revision, words, heads and seeds.

**How to use it**
1. In Colab, select **Runtime → Change runtime type → GPU**.
2. For gated checkpoints (Llama), accept the licence on Hugging Face and store a read token in Colab Secrets as
   **HF_TOKEN**. The default Qwen model needs no token.
3. Edit the two settings cells and choose **Run all**. The last cell shows the figures and downloads a ZIP of results.

| GPU | Starting point in native 16-bit precision |
|---|---|
| T4, 16 GB | Llama 3.2 1B (FP16) or Qwen2.5-1.5B (FP32, see the optional settings). Start with a short walk. |
| L4, 24 GB | Llama 3.1 8B, Mistral 7B or Qwen2.5-7B. The notebook chooses BF16. |
| A100, 40 or 80 GB | Qwen2.5-14B, or a longer experiment on a smaller checkpoint. |

These are starting points: walk length, attention workspace and free memory decide what fits. Model download time comes
on top of the experiment's run time.

**About the code.** The ablations run in a compact re-implementation of the transformer forward pass that reuses each
walk's history across conditions (section 3). Before every run, the notebook compares it with ordinary Hugging Face
forward passes (section 4). Software versions, model revisions, seeds and the tests of the code are listed under
*Technical details and reproducibility* at the end. The notebook was generated with AI models (GPT-6.2 Astra and Claude Opus 5.5) following the author's instructions, and checked by the author.

## 1. Choose the experiment

| Topology | Size control | Shape |
|---|---|---|
| Square | Rows × columns of vertices | Open boundaries, horizontal and vertical edges. Default 4×4 (16 vertices). |
| Ring | Number of vertices | A cycle. Default 10 vertices. |
| Hexagonal | Rows × columns of hexagonal **cells** | Vertices on the cell corners. The default 3×3 patch has 30 vertices and 38 edges; Park et al.'s hexagonal lattice also has 30 vertices (their Fig. 4c). |
| Triangular | Rows × columns of vertices | The square grid with the same diagonal added to every square: a parallelogram of the triangular lattice (a 60° rhombus when rows = columns), up to six neighbours per vertex. Default 4×4. |
| Random | Number of vertices and extra-edge probability | A uniformly random labelled spanning tree, plus each missing edge independently with the chosen probability. Default 16 vertices. |

Set a size field to **0** to use its topology's default. Every topology uses the same walk: at each step the walk moves
to a uniformly random neighbour, and it may step straight back. (For the ring, Park et al. instead sampled independent
pairs of neighbouring words; one walk rule for all graphs keeps them comparable.)

**Head counts.** `1,2,4,8,16,32` is Arditi's ladder, `1-32` gives every count, and a short list such as `4,16` is
quicker. Each count k runs the top k induction heads, the top k previous-token heads, and the **union** of both (a head
on both lists is removed once). The intact model is the shared k = 0 baseline.

**Controls.**
- `RANDOM_CONTROL_DRAWS`: random heads drawn from all heads outside both top-32 lists, as many as the largest groups
  remove. For a single family that is the largest k; for the union it is the number of distinct heads in the two top-k
  lists at the largest k (a head on both lists counts once), so up to twice as many. These controls are matched in size
  to the largest k only. With Arditi's ladder (largest k = 32) and 4 draws, the single-family control follows his
  procedure: 32 heads, averaged over 4 independent draws.
- `LAYER_MATCHED_DRAWS`: for every head group, the same number of random heads from the **same layers** (again outside
  both top-32 lists). Removing a head changes everything computed after its layer, so this is the fairer control when
  comparing scopes. Each draw adds as many conditions as the head groups themselves.

**To reproduce Arditi's setup:** square 4×4, 1,400 words, 16 walks (one from each starting vertex), head counts
`1,2,4,8,16,32`, 4 random draws, `meta-llama/Llama-3.1-8B` with automatic head rankings. The walks are new random paths
drawn from `SEED`. The word count does not include the beginning-of-sequence token.

In [ ]:
#@title 1 · Choose a model, graph and experiment size
MODEL_ID = "Qwen/Qwen2.5-1.5B" #@param ["Qwen/Qwen2.5-1.5B","Qwen/Qwen2.5-3B","Qwen/Qwen2.5-7B","Qwen/Qwen2.5-14B","meta-llama/Llama-3.2-1B","meta-llama/Llama-3.2-3B","meta-llama/Llama-3.1-8B","mistralai/Mistral-7B-v0.3"]
TOPOLOGY = "square" #@param ["square","ring","hexagonal","triangular","random"]
ROWS = 0 #@param {type:"integer"}
COLUMNS = 0 #@param {type:"integer"}
NUMBER_OF_NODES = 0 #@param {type:"integer"}
RANDOM_EDGE_PROBABILITY = 0.15 #@param {type:"number"}
NUMBER_OF_SEQUENCES = 4 #@param {type:"integer"}
NUMBER_OF_TOKENS = 512 #@param {type:"integer"}
HEAD_COUNTS = "1,2,4,8,16,32" #@param {type:"string"}
RANDOM_CONTROL_DRAWS = 1 #@param [0,1,4] {type:"raw"}
LAYER_MATCHED_DRAWS = 0 #@param [0,1,4] {type:"raw"}
SEED = 2025 #@param {type:"integer"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}

**Optional settings.**
- `ACCURACY_STRIDE`: 1 scores every word; a larger stride scores fewer positions (PCA still uses every position of each
  window).
- `SUMMARY_FROM_WORD`: first word of the window used for the summary table; 0 = the second half of the walk.
- `PCA_LENGTHS`: comma-separated context lengths for the PCA maps, or `auto`. `PCA_WINDOW`: how many words before each
  length are averaged per word.
- `PCA_LAYER`: −1 takes the residual stream entering the block closest to 26/32 of the model's depth (block 26 of
  Llama 3.1 8B, the layer Park et al. used; block 13 of Llama 3.2 1B). Block indices start at zero.
- `HEAD_SOURCE`: `auto` uses stored head rankings where they exist (section 2) and the probe otherwise; `recalibrate`
  always runs the probe.
- `WORD_ASSIGNMENT_SEED`: −1 takes the first suitable words of the list in order, which puts Park et al.'s 16 words in
  their layout on the 4×4 grid; a value of 0 or more draws the words at random, to check how much the results depend on
  them. Custom words override it.
- `QUERY_CHUNK` bounds the number of independent queries in one attention block and `LOGIT_ROWS` the number of
  full-vocabulary predictions computed together. Lower either one on a tight GPU.
- `CUSTOM_WORDS`: optional, one distinct single-token word per vertex, comma-separated, in vertex order.
- `PRECISION`: `auto` uses bfloat16 on GPUs that support it (L4, A100). On older GPUs such as the T4 it uses float16,
  except for Qwen models, whose activations can exceed the float16 range (Qwen recommends bfloat16): they run in
  float32, which doubles the memory of the weights and is slower (Qwen2.5-3B in float32 barely fits on a T4). Choose a
  precision explicitly if the notebook reports overflow. The precision is part of the run's identity.

In [ ]:
#@title 2 · Optional measurement and memory settings
ACCURACY_STRIDE = 1 #@param {type:"integer"}
SUMMARY_FROM_WORD = 0 #@param {type:"integer"}
PCA_LENGTHS = "auto" #@param {type:"string"}
PCA_WINDOW = 200 #@param {type:"integer"}
PCA_LAYER = -1 #@param {type:"integer"}
HEAD_SOURCE = "auto" #@param ["auto","recalibrate"]
WORD_ASSIGNMENT_SEED = -1 #@param {type:"integer"}
QUERY_CHUNK = 0 #@param {type:"integer"}
LOGIT_ROWS = 32 #@param {type:"integer"}
RUN_NATIVE_CHECK = True #@param {type:"boolean"}
PRECISION = "auto" #@param ["auto","float32","float16","bfloat16"]
CUSTOM_WORDS = "" #@param {type:"string"}

In [ ]:
#@title Install the tested Transformers version and notebook dependencies
import importlib.metadata as metadata
import subprocess
import sys
from packaging.requirements import Requirement

requirements = ["transformers==4.56.2", "accelerate==1.10.1", "networkx>=3.2,<4", "matplotlib>=3.6",
                "sentencepiece>=0.1.99", "filelock>=3", "tqdm>=4", "pandas>=1.5"]
needed = []
for spec in requirements:
    requirement = Requirement(spec)
    try:
        present = metadata.version(requirement.name)
    except metadata.PackageNotFoundError:
        present = None
    if present is None or present not in requirement.specifier:
        needed.append(spec)
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *needed])
for package in ("transformers", "accelerate", "networkx"):
    if package in sys.modules and sys.modules[package].__version__ != metadata.version(package):
        raise RuntimeError("Restart the Colab session once to activate the installed version, then run all cells.")
print("Transformers", metadata.version("transformers"), "· PyTorch", metadata.version("torch"))

## 2. Build the graph and identify heads

The graph and the walks use separate deterministic random streams, so every ablation condition sees exactly the same
words in the same order. Each walk begins with the tokenizer's beginning-of-sequence token when it has one. The word
IDs are the tokens of each word *after a space*, including the first word, and the notebook checks the word boundaries
of every walk.

**Head rankings.** The probe follows Arditi (Appendix B of the blog post): 32 sequences of 32 random tokens from the
lower half of the vocabulary, each repeated twice. A head's *previous-token score* is its average attention from each
position to the one before; its *induction score* is its average attention, in the second half, to the position just
after the earlier copy of the current token. Heads are ranked separately by each score. The probe runs through the
notebook's engine (section 3) and recomputes every attention probability in float32 from the layer's queries and keys,
so that 16-bit models cannot overflow in the explicit attention scores. With `HEAD_SOURCE = "auto"`,
two stored rankings replace the probe:
- **Llama 3.1 8B** at the pinned revision: the head scores released with Arditi's code, so that results are comparable
  with his;
- **Llama 3.2 1B**, only if its weight file is byte-identical to the one used before: a ranking measured with the same
  probe in float32 for the author's earlier experiments, so that results are comparable with those.

Every other model is ranked by the probe on the loaded checkpoint. Choose `recalibrate` to use the probe everywhere.

In [ ]:
#@title Model and measurement helpers
import gc
import hashlib
import json
import math
import shutil
import time
from pathlib import Path

import numpy as np
import torch
import transformers
from huggingface_hub import get_hf_file_metadata, get_token, hf_hub_url
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer


PUBLISHED_8B_REVISION = "d04e592bb4f6aa9cfee91e2e20afa771667e1d4b"
PRIOR_1B_WEIGHT_SHA = "68a2e4be76fa709455a60272fba8e512c02d81c46e6c671cc9449e374fd6809a"


def hub_access(model_id):
    """Use an existing login, or a Colab secret for repositories that may need access."""
    token = get_token()
    if token or model_id.startswith("Qwen/"):
        return token
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except (ImportError, KeyError):
        return None
    except Exception as exc:
        # Colab raises its own named exceptions when a secret is absent or unavailable.
        if type(exc).__name__ in {"SecretNotFoundError", "NotebookAccessError"}:
            return None
        raise


def choose_dtype(model_id, device, precision="auto"):
    """bfloat16 on Ampere or newer GPUs (L4, A100). Older GPUs such as the T4 have no native bfloat16: there the
    notebook uses float16, except for Qwen models, whose activations can exceed the float16 range (Qwen recommends
    bfloat16), so they run in float32. An explicit PRECISION overrides the choice."""
    if precision != "auto":
        return {"float32": torch.float32, "float16": torch.float16, "bfloat16": torch.bfloat16}[precision]
    if torch.cuda.get_device_capability(device)[0] >= 8:
        return torch.bfloat16
    return torch.float32 if model_id.startswith("Qwen/") else torch.float16


def load_checkpoint(model_id, precision="auto"):
    if not torch.cuda.is_available():
        raise RuntimeError("In Colab, choose Runtime → Change runtime type → GPU and reconnect.")
    device = torch.device("cuda:0")
    dtype = choose_dtype(model_id, device, precision)
    if dtype == torch.bfloat16 and torch.cuda.get_device_capability(device)[0] < 8:
        print("This GPU has no native bfloat16: it is emulated, slowly, and some operations may be unsupported.")
    token = hub_access(model_id)
    requested = PUBLISHED_8B_REVISION if model_id == "meta-llama/Llama-3.1-8B" else "main"
    config = AutoConfig.from_pretrained(model_id, token=token, revision=requested)
    revision = config._commit_hash or requested
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token, revision=revision)
    print(f"Loading {model_id} on {torch.cuda.get_device_name(device)} in {dtype}.", flush=True)
    network = AutoModelForCausalLM.from_pretrained(
        model_id, config=config, revision=revision, token=token, dtype=dtype,
        device_map={"": 0}, use_safetensors=True, low_cpu_mem_usage=True, attn_implementation="sdpa",
    ).eval().requires_grad_(False)
    same_1b = False
    if model_id == "meta-llama/Llama-3.2-1B":
        # Only reuse the earlier 1B ranking if the actual weights have the same fingerprint.
        try:
            url = hf_hub_url(model_id, "model.safetensors", revision=revision)
            same_1b = get_hf_file_metadata(url, token=token).etag == PRIOR_1B_WEIGHT_SHA
        except OSError:
            print("Weight fingerprint unavailable. The notebook will calibrate the heads locally.")
    return tokenizer, network, revision, same_1b


def choose_cutoffs(length, text):
    if text.strip().lower() == "auto":
        candidates = [1, 4, 16, 32, 100, 250, 500, 1000, 1500, 3000, 6000, 8000]
        return sorted({t for t in candidates if t < length} | {length})
    chosen = {int(t.strip()) for t in text.split(",")}
    if not chosen or min(chosen) < 1 or max(chosen) > length:
        raise ValueError("Each PCA length must be between 1 and NUMBER_OF_TOKENS.")
    return sorted(chosen)


def choose_probe_layer(depth, requested):
    if requested == -1:
        return min(depth - 1, round(26 * depth / 32))
    if not 0 <= requested < depth:
        raise ValueError(f"PCA_LAYER must be -1 or a zero-based block index between 0 and {depth - 1}.")
    return requested


def choose_query_chunk(requested):
    if requested:
        return requested
    free_gib = torch.cuda.mem_get_info()[0] / 2**30
    return 32 if free_gib < 5 else (64 if free_gib < 15 else 128)


def write_json(path, data):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)

In [ ]:
#@title Graph constructors, word selection and reproducible walks
# Graph vertices are numbered 0, 1, ...; those numbers index the word list.
import numpy as np
import networkx as nx


def build_graph(topology, rows=4, cols=4, n_nodes=16, random_edge_probability=0.15, seed=2025):
    """Return a boolean adjacency matrix and one reference drawing position per node."""
    if topology in ("square", "triangular", "hexagonal"):
        if rows < 1 or cols < 1:
            raise ValueError("Rows and columns must be positive.")
        if topology == "hexagonal":
            # Nodes sit on the corners of hexagons. A 3 × 3 patch has 30 nodes.
            graph = nx.hexagonal_lattice_graph(rows, cols)
            positions = nx.get_node_attributes(graph, "pos")
        else:
            graph = nx.grid_2d_graph(rows, cols)
            positions = {(r, c): (c, -r) for r, c in graph}
            if topology == "triangular":
                # Add one diagonal per square, then shear the drawing so all edges have length 1.
                for r in range(rows - 1):
                    for c in range(1, cols):
                        graph.add_edge((r, c), (r + 1, c - 1))
                positions = {(r, c): (c + r / 2, -np.sqrt(3) * r / 2) for r, c in graph}
    elif topology == "ring":
        if n_nodes < 3:
            raise ValueError("A ring needs at least three nodes.")
        graph = nx.cycle_graph(n_nodes)
        positions = nx.circular_layout(graph)
    elif topology == "random":
        if n_nodes < 2 or not 0 <= random_edge_probability <= 1:
            raise ValueError("A random graph needs at least two nodes and an extra-edge probability in [0, 1].")
        # A uniform labeled spanning tree keeps the graph connected even when p = 0.
        # Independently add each remaining edge with probability p.
        rng = np.random.default_rng(seed)
        prufer_code = rng.integers(n_nodes, size=n_nodes - 2).tolist()
        graph = nx.from_prufer_sequence(prufer_code)
        extra_edges = nx.fast_gnp_random_graph(n_nodes, random_edge_probability, seed=rng)
        graph.add_edges_from(extra_edges.edges())
        positions = nx.spring_layout(graph, seed=int(rng.integers(2**32)))
    else:
        raise ValueError("Choose square, ring, hexagonal, triangular, or random.")

    if len(graph) < 2:
        raise ValueError("The graph needs at least two nodes to take a step.")
    # Sorting (row, column) pairs gives the same square board numbering as Arditi.
    nodes = sorted(graph)
    adjacency = nx.to_numpy_array(graph, nodelist=nodes, dtype=bool)
    coordinates = np.array([positions[node] for node in nodes], dtype=float)
    return adjacency, coordinates


# Park et al.'s 16 words (also used by Arditi) come first, in the order of Park et al.'s Fig. 1, so that a 4×4 grid
# gets exactly their layout. The other nouns serve larger graphs and tokenizers that split some words into pieces.
# A custom word list is used in the user's exact node order.
NOUN_POOL = """
apple bird car egg house milk plane opera box sand sun mango rock math code
phone drum chair lake onion glove star key towel horse rice coin wall plum book
boat shell hammer desk river flour cloud lamp pear train salt shoe drumstick fence ant
bread castle bell straw grape door fox kite silk ball broom arch jewel carpet cup
roof glass spoon fern frog clock cane tin fish wind ring peach twig brush nest
stone singer bench pipe wolf hoop lemon pen oak cart wire chess goat page hill
mail vase goose wheel pond cheese brick harp root oat yard dune rack lily bull
sugar spoonbill snail flame flag crane stove tree raft dock gear bed comb yoke chalk
note rope pencil swan scissors lock crow tea rail wheat soap mint spring cross shark
urn map harpist cap hinge leaf flute lens bag slope hive rose iron starfish screw
curtain chalkboard rug gum corn chain barge bone pan nut pump shore beak torch seal
mitten kelp float tray coat beam daisy boot fork paint loaf ruler belt pearl bucket
chestnut yardstick jug medal ink cat alley shawl sponge lentil axe club motor jar cabin
birch sugarcane cake raven pupil cuff seed mirror dish bath well clam table fog kayak
zipper eagle ice signal badge moss ballad eel earth angel sandbank chest tart wasp silver
sword frost maple planet whale trumpet mushroom tail bellows lion kitten collar comet trowel berry
spade crystal field bear wafer sled copper radish queen wax moon hair lizard letter sail
stem feather walnut saddle ladder ticket cactus pillow kettle basket garden paper sleeve cub ladle
market thread rabbit switch button kernel orange statue diesel chimney cotton camera parrot apron canvas
magnet pickle daffodil compass pollen napkin window turtle marble orchid tulip sailor violin helmet oyster
rubber flask pantry piston pepper uncle plumber tomato cable bike anchor canyon island granite scarf
pedal pocket mower engine tiger apricot barber rocket crater ivy village thunder hedgehog spout shutter
wagon needle valley timber forearm handle tunnel scalp buckle cricket opal bead cellar windmill parsnip
goblet paddle carrot bubble kneecap syrup plank giant spider meadow bottle acorn noodle shovel sack
periscope bass shed fuse horn drone bank kiln canoe moth duffel suitcase gooseberry stew breeze
skin pool caper atlas dandelion sofa rod mole petal silo bakery parsley beetle sock mud
cottage velvet lightning coaster tower locket millet cedar brook palm plaster bamboo violinist barley axle
claw gardenia tank coffee slab bat eyelid runway silverware chapel traffic chip linen ocean hoof
tube pulse camel cave lava riverbed tripod knife magazine radio owl cereal barrel fossil chapter
wing bonnet clover nail coral referee moat theater lobster card ferry china arrow gourd candlestick
wave cattle socket film spine mask cinder ship glacier jelly keyboard parka cove
""".split()


def choose_node_words(tokenizer, n_nodes, custom_words=None, word_seed=-1):
    """Find distinct, ordinary word IDs as they occur *after* text in a sequence.
    word_seed = -1 takes the first suitable words of the pool in order (Park et al.'s 16 words on a 4×4 grid);
    word_seed >= 0 draws n_nodes words at random from all suitable words of the pool. Custom words are used as given."""
    custom = custom_words is not None and len(custom_words) > 0
    candidates = NOUN_POOL
    if custom:
        candidates = custom_words
        if isinstance(custom_words, str):
            candidates = custom_words.replace(",", " ").split()
        if len(candidates) != n_nodes:
            raise ValueError(f"Supply exactly {n_nodes} custom words, one per node.")

    # This anchor avoids the special first-word treatment in SentencePiece tokenizers.
    anchor = tokenizer.encode("A", add_special_tokens=False)
    reserved = set(tokenizer.all_special_ids)
    words, token_ids = [], []
    for word in candidates:
        if not isinstance(word, str) or not word.isascii() or not word.isalpha():
            raise ValueError(f"Use a single English word for each node; received {word!r}.")
        encoded = tokenizer.encode("A " + word, add_special_tokens=False)
        valid = len(encoded) == len(anchor) + 1 and encoded[:-1] == anchor
        if valid and encoded[-1] not in reserved and encoded[-1] not in token_ids:
            words.append(word)
            token_ids.append(encoded[-1])
        elif custom:
            raise ValueError(f"{word!r} needs a unique, non-special token after a space in this tokenizer.")
        if len(words) == n_nodes and (custom or word_seed < 0):
            return words, np.array(token_ids, dtype=np.int64)
    if not custom and word_seed >= 0 and len(words) >= n_nodes:
        pick = np.random.default_rng(word_seed).choice(len(words), size=n_nodes, replace=False)
        return [words[i] for i in pick], np.array(token_ids, dtype=np.int64)[pick]
    raise ValueError(f"This tokenizer accepted {len(words)} built-in nouns; the graph needs {n_nodes}. "
                     "Use a smaller graph or supply a custom list of single-token nouns.")


def walk_nodes(adjacency, n_tokens, sequence_index, seed=2025):
    """Take uniform steps to neighboring vertices. Backtracking is allowed."""
    if n_tokens < 1 or sequence_index < 0:
        raise ValueError("Use a positive token count and a nonnegative sequence index.")
    size = len(adjacency)
    cycle, offset = divmod(sequence_index, size)
    if size == 16 and cycle == 0:
        # For 16 vertices the first four starts are 0, 2, 5 and 11, the starts of earlier runs of this experiment;
        # the other starts follow.
        order = [0, 2, 5, 11, 1, 3, 4, 6, 7, 8, 9, 10, 12, 13, 14, 15]
    else:
        order_rng = np.random.default_rng(np.random.SeedSequence(seed, spawn_key=(2**32 - 1, cycle)))
        order = order_rng.permutation(size)
    start = int(order[offset])
    # Each batch of size walks uses every start once. Repeated batches get fresh paths.
    stream = np.random.SeedSequence(seed, spawn_key=(cycle * size + start,))
    rng = np.random.default_rng(stream)
    neighbors = [np.flatnonzero(row) for row in adjacency]
    path = np.empty(n_tokens, dtype=np.int32)
    path[0] = start
    for step in range(1, n_tokens):
        path[step] = rng.choice(neighbors[path[step - 1]])
    return path


def verify_walk_tokens(tokenizer, words, word_ids, nodes):
    """Check every word boundary of the exact walk before running the model."""
    text = "A " + " ".join(words[i] for i in nodes)
    observed = tokenizer.encode(text, add_special_tokens=False, verbose=False)
    anchor = tokenizer.encode("A", add_special_tokens=False)
    expected = anchor + np.asarray(word_ids)[nodes].tolist()
    if observed != expected:
        raise ValueError("The tokenizer changed a word ID or word boundary in this walk. "
                         "Choose different node words for this model.")

In [ ]:
#@title Stored Llama head rankings
# Each pair is (zero-based block, query head), strongest first.
# Llama 3.1 8B: from the head scores released with Arditi's code, for the pinned checkpoint revision.
# Llama 3.2 1B: measured with the probe below, in float32, for the author's earlier experiments; used only when the
# loaded weight file is byte-identical to the one measured.
REFERENCE_RANKINGS = {
    'meta-llama/Llama-3.1-8B': {
        'induction': [
            (15, 30), (8, 1), (16, 20), (2, 22), (10, 14), (5, 11), (15, 1), (24, 27),
            (20, 1), (5, 8), (20, 14), (19, 3), (26, 15), (2, 20), (27, 6), (2, 12),
            (26, 13), (28, 15), (16, 1), (27, 5), (13, 6), (2, 25), (27, 7), (22, 14),
            (16, 23), (27, 4), (5, 9), (20, 13), (10, 13), (2, 21), (15, 28), (19, 0),
        ],
        'previous': [
            (0, 2), (14, 26), (1, 20), (9, 11), (1, 18), (0, 3), (1, 16), (14, 8),
            (7, 2), (0, 24), (1, 13), (0, 26), (21, 7), (11, 16), (0, 6), (7, 1),
            (1, 4), (6, 8), (12, 16), (1, 26), (9, 10), (25, 20), (1, 24), (16, 29),
            (18, 26), (9, 8), (21, 10), (14, 9), (11, 19), (1, 30), (16, 22), (1, 1),
        ],
    },
    'meta-llama/Llama-3.2-1B': {
        'induction': [
            (10, 23), (5, 10), (12, 15), (2, 26), (12, 13), (10, 1), (10, 20), (13, 20),
            (12, 2), (7, 5), (15, 12), (5, 8), (2, 20), (5, 11), (14, 20), (10, 21),
            (5, 9), (14, 21), (13, 21), (10, 22), (15, 20), (2, 14), (13, 2), (10, 13),
            (14, 22), (14, 15), (14, 29), (11, 20), (14, 9), (11, 22), (7, 6), (11, 4),
        ],
        'previous': [
            (0, 2), (8, 26), (1, 20), (9, 28), (11, 11), (0, 24), (4, 12), (0, 3),
            (1, 16), (6, 3), (6, 0), (3, 17), (1, 18), (0, 26), (13, 27), (0, 6),
            (1, 26), (0, 7), (6, 2), (0, 5), (1, 24), (0, 27), (4, 13), (0, 4),
            (0, 1), (8, 9), (5, 13), (1, 1), (1, 4), (3, 18), (0, 25), (8, 11),
        ],
    },
}

In [ ]:
#@title Measure attention patterns and construct the head groups
def parse_head_counts(text):
    """Accept either a comma-separated ladder or inclusive ranges such as 1-32."""
    counts = set()
    for entry in text.split(","):
        ends = [int(item.strip()) for item in entry.split("-")]
        if len(ends) > 2 or not 1 <= ends[0] <= ends[-1] <= 32:
            raise ValueError("Enter head counts from 1 to 32, e.g. '1,2,4,8,16,32' or '1-32'.")
        counts.update(range(ends[0], ends[-1] + 1))
    return sorted(counts)


@torch.inference_mode()
def measure_head_rankings(engine, probe_sequences=32, seed=37139):
    """Repeat 32 random token IDs twice. Rank query heads by attention on two diagonals.
    The forward pass is the notebook's engine (section 3), and each layer's attention probabilities are recomputed in
    float32 from its queries and keys: a 16-bit model's explicit attention scores could otherwise overflow."""
    config = engine.config
    device = engine.device
    random = np.random.default_rng(seed)
    first_half = random.integers(0, config.vocab_size // 2, size=(probe_sequences, 32))
    prompts = torch.as_tensor(np.tile(first_half, (1, 2)), device=device)
    length = prompts.shape[1]
    induction_q = torch.arange(32, 64, device=device)
    previous_q = torch.arange(1, 64, device=device)
    shape = (config.num_hidden_layers, config.num_attention_heads)
    induction = torch.zeros(shape, device=device, dtype=torch.float32)
    previous = torch.zeros_like(induction)
    future = torch.ones(length, length, dtype=torch.bool, device=device).triu(1)    # keys a query may not see
    positions = torch.arange(length, device=device)[None, :]

    for prompt in prompts:
        hidden = engine.base.embed_tokens(prompt)
        angles = tuple(a[0, :, None] for a in engine.base.rotary_emb(hidden[None], positions))
        for layer, block in enumerate(engine.layers):
            query, key, value = engine._project(hidden, block, angles)        # [n, H, d]; [K, n, d]; [K, n, d]
            keys = key.repeat_interleave(engine.heads_per_key, dim=0).float()  # [H, n, d]
            scores = torch.einsum("nhd,hmd->hnm", query.float(), keys) * block.self_attn.scaling
            weights = scores.masked_fill(future, -torch.inf).softmax(dim=-1)   # [head, query, key]
            # An induction head reads the successor of the matching token in the first half.
            induction[layer] += weights[:, induction_q, induction_q - 31].mean(-1)
            previous[layer] += weights[:, previous_q, previous_q - 1].mean(-1)
            reads = engine._full_attention(query, key, value, block.self_attn.scaling, engine.windows[layer])
            hidden = engine._finish(hidden, block, reads, None)

    rankings = {}
    for name, scores in (("induction", induction), ("previous", previous)):
        scores = (scores / probe_sequences).cpu().numpy()
        if not np.isfinite(scores).all():
            raise FloatingPointError(
                f"The repeated-token probe produced a nonfinite head score: the model's activations overflow in "
                f"{engine.model.dtype}. Set PRECISION = 'float32' in the optional settings, or use a GPU with "
                "bfloat16 (L4, A100), then run all cells again.")
        flat = np.argsort(-scores.ravel(), kind="stable")[:32]
        rankings[name] = [tuple(map(int, np.unravel_index(i, shape))) for i in flat]
    return rankings


def choose_head_rankings(engine, model_id, revision, source, reference_1b_verified=False):
    if source == "auto":
        use_published = model_id == "meta-llama/Llama-3.1-8B" and revision == PUBLISHED_8B_REVISION
        use_prior_1b = model_id == "meta-llama/Llama-3.2-1B" and reference_1b_verified
        if use_published or use_prior_1b:
            origin = ("Arditi (2026), released 8B head scores" if use_published
                      else "Stored float32 ranking of Llama 3.2 1B (identical weight file)")
            return REFERENCE_RANKINGS[model_id], origin
    return measure_head_rankings(engine), "Repeated-token probe on selected checkpoint: 32 prompts, seed 37139"


def make_conditions(rankings, counts, random_draws, layer_count, head_count, seed, matched_draws=0):
    """Map each intervention to its query heads and to the three places it can be applied."""
    groups = {}
    conditions = [dict(name="intact", family="intact", k=0, scope="intact")]

    def add(name, family, k, pairs, **extra):
        groups[name] = sorted(set(map(tuple, pairs)))
        for scope in ("everywhere", "history", "final"):
            conditions.append(dict(name=f"{name}_{scope}", family=family, k=k, scope=scope, **extra))

    targets = {}
    for k in counts:
        induction = list(rankings["induction"][:k])
        previous = list(rankings["previous"][:k])
        targets[k] = {"induction": induction, "previous": previous, "both": induction + previous}
        add(f"induction{k}", "induction", k, induction)
        add(f"previous{k}", "previous", k, previous)
        add(f"both{k}", "both", k, induction + previous)

    excluded = set(map(tuple, rankings["induction"][:32])) | set(map(tuple, rankings["previous"][:32]))
    eligible = [(layer, head) for layer in range(layer_count) for head in range(head_count)
                if (layer, head) not in excluded]
    # Random controls are as large as the largest groups: k_max heads for a single family, and for the union as many
    # heads as the two top-k_max lists remove together (a head on both lists counts once). Each draw has its own seed,
    # so four draws keep the first one; with k_max = 32 the single-family draws are Arditi's 32-head controls.
    k_max = max(counts)
    union_size = len(groups[f"both{k_max}"])
    if random_draws and len(eligible) < union_size:
        raise ValueError("The checkpoint has too few heads outside both top-32 lists for the random controls.")
    for draw, child_seed in enumerate(np.random.SeedSequence(seed).spawn(random_draws)):
        indices = np.random.default_rng(child_seed).choice(len(eligible), size=k_max, replace=False)
        add(f"random{k_max}_draw{draw + 1}", "random", k_max, [eligible[i] for i in indices],
            matches="single", matched_k=k_max)
    for draw, child_seed in enumerate(np.random.SeedSequence([seed, 2]).spawn(random_draws)):
        indices = np.random.default_rng(child_seed).choice(len(eligible), size=union_size, replace=False)
        add(f"random{union_size}union_draw{draw + 1}", "random", union_size, [eligible[i] for i in indices],
            matches="union", matched_k=k_max)

    # Layer-matched controls: for every head group, one random head from the same layer for each head of the group,
    # outside both top-32 lists. A separate random stream keeps the draws above unchanged.
    if matched_draws:
        by_layer = {}
        for pair in eligible:
            by_layer.setdefault(pair[0], []).append(pair)
        for draw, child_seed in enumerate(np.random.SeedSequence([seed, 1]).spawn(matched_draws)):
            draw_rng = np.random.default_rng(child_seed)
            for k in counts:
                for family in ("induction", "previous", "both"):
                    chosen = []
                    for layer, _ in sorted(set(map(tuple, targets[k][family]))):
                        pool = [pair for pair in by_layer.get(layer, []) if pair not in chosen]
                        if not pool:      # every eligible head of this layer is taken: use any eligible head
                            pool = [pair for pair in eligible if pair not in chosen]
                        chosen.append(pool[draw_rng.integers(len(pool))])
                    add(f"{family}{k}_matched{draw + 1}", f"{family}_matched", k, chosen)
    return groups, conditions

In [ ]:
#@title Check the requested experiment before loading the model
for name, count in {"NUMBER_OF_SEQUENCES": NUMBER_OF_SEQUENCES, "NUMBER_OF_TOKENS": NUMBER_OF_TOKENS,
                    "ACCURACY_STRIDE": ACCURACY_STRIDE, "PCA_WINDOW": PCA_WINDOW, "LOGIT_ROWS": LOGIT_ROWS}.items():
    if type(count) is not int or count < 1:
        raise ValueError(f"{name} must be a positive integer.")
for name, value in {"ROWS": ROWS, "COLUMNS": COLUMNS, "NUMBER_OF_NODES": NUMBER_OF_NODES,
                    "QUERY_CHUNK": QUERY_CHUNK, "SEED": SEED, "SUMMARY_FROM_WORD": SUMMARY_FROM_WORD}.items():
    if type(value) is not int or value < 0:
        raise ValueError(f"{name} must be a nonnegative integer.")
if type(WORD_ASSIGNMENT_SEED) is not int or WORD_ASSIGNMENT_SEED < -1:
    raise ValueError("WORD_ASSIGNMENT_SEED must be -1 (list order) or a nonnegative integer.")
if (RANDOM_CONTROL_DRAWS not in (0, 1, 4) or LAYER_MATCHED_DRAWS not in (0, 1, 4)
        or HEAD_SOURCE not in ("auto", "recalibrate")):
    raise ValueError("Select a listed head source and listed control counts.")
if not 0 <= RANDOM_EDGE_PROBABILITY <= 1:
    raise ValueError("Choose a random edge probability between 0 and 1.")
if PRECISION not in ("auto", "float32", "float16", "bfloat16"):
    raise ValueError("PRECISION must be auto, float32, float16 or bfloat16.")
if SUMMARY_FROM_WORD > NUMBER_OF_TOKENS:
    raise ValueError("SUMMARY_FROM_WORD must not exceed NUMBER_OF_TOKENS.")
rows = ROWS or (3 if TOPOLOGY == "hexagonal" else 4)
columns = COLUMNS or (3 if TOPOLOGY == "hexagonal" else 4)
requested_nodes = NUMBER_OF_NODES or (10 if TOPOLOGY == "ring" else 16)
planned = rows * columns if TOPOLOGY in ("square", "triangular") else requested_nodes
if TOPOLOGY == "hexagonal":
    planned = 2 * (rows * columns + rows + columns)
if not CUSTOM_WORDS.strip() and planned > len(NOUN_POOL):
    raise ValueError(f"The graph needs {planned} labels. Supply custom words or reduce its size "
                     f"to use the built-in pool of {len(NOUN_POOL)} candidate nouns.")
if CUSTOM_WORDS.strip() and len(CUSTOM_WORDS.split(",")) != planned:
    raise ValueError(f"The requested graph needs exactly {planned} comma-separated custom words.")
adjacency, graph_xy = build_graph(TOPOLOGY, rows, columns, requested_nodes, RANDOM_EDGE_PROBABILITY, SEED)
head_counts = parse_head_counts(HEAD_COUNTS)
cutoffs = choose_cutoffs(NUMBER_OF_TOKENS, PCA_LENGTHS)
score_at = np.unique(np.r_[np.arange(1, NUMBER_OF_TOKENS + 1, ACCURACY_STRIDE), NUMBER_OF_TOKENS])
summary_from = SUMMARY_FROM_WORD or NUMBER_OF_TOKENS // 2 + 1
graph_name = f"{TOPOLOGY} {rows}×{columns}" if TOPOLOGY in ("square", "hexagonal", "triangular") else TOPOLOGY
n_groups = 3 * len(head_counts) * (1 + LAYER_MATCHED_DRAWS) + 2 * RANDOM_CONTROL_DRAWS
print(f"{graph_name}: {len(adjacency)} vertices, {int(adjacency.sum()) // 2} undirected edges.")
print(f"{NUMBER_OF_SEQUENCES} walks × {NUMBER_OF_TOKENS} words. Accuracy at {len(score_at)} lengths; PCA at {cutoffs}.")
print(f"{1 + 3 * n_groups} conditions per walk, including intact. Summary window: words {summary_from}–{NUMBER_OF_TOKENS}.")

## 3. Reuse each history

| Condition | Earlier positions | Position that predicts the next word |
|---|---|---|
| Intact | All heads active | All heads active |
| Final word only | Computed normally | Selected heads zeroed |
| Everywhere | Selected heads zeroed | Selected heads zeroed |
| History only | Selected heads zeroed | All heads active |

**Zero ablation.** A head is switched off by setting its output to zero just before the attention output projection,
as in Arditi's experiment. This removes the selected *query* heads, even when several heads share keys and values; the
MLPs and all other heads keep working. Zeroing can push activations away from their usual range, so a drop measures how
much the model relies on a head's normal output, not exactly what that head computes.

**How the engine saves time.** For each walk it runs one intact causal pass and keeps the keys and values of the
history. Every final-word-only condition reuses that history and recomputes only the scored positions through the
affected layers. For each head group it then runs one ablated pass (the everywhere condition) and reuses its history for
the history-only condition. Each scored position is recomputed with its own key and value only, so different positions
can be evaluated together without seeing each other. Only one multi-layer history cache is kept at a time.

**Which positions count as "the final word".** Every scored position, and every position used for the PCA maps, is
treated as if it were the last word of the context. In the final-word-only maps, for example, each position's hidden
state is computed with the heads zeroed at that position only.

In [ ]:
#@title Transformer engine · prefill, independent endpoint replay and readout
"""One full prefix per intervention; each alternative endpoint sees its own private state."""
import numpy as np
import torch
import torch.nn.functional as F


class WalkReadout:
    """Choose which positions need logits and which contribute to PCA word means."""

    def __init__(self, nodes, board_ids, adjacency, cutoffs, window, positions, prefix_length):
        self.nodes = np.asarray(nodes, dtype=np.int64)
        self.cutoffs = np.asarray(cutoffs, dtype=np.int64)
        self.board_ids = np.asarray(board_ids, dtype=np.int64)
        self.adjacency = np.asarray(adjacency, dtype=bool)
        self.window = int(window)
        n = len(self.nodes)
        if positions is None:
            positions = np.arange(1, n + 1)
        self.scored = np.asarray(positions, dtype=np.int64)
        if n == 0 or self.window < 1 or prefix_length < 0:
            raise ValueError("The walk, window, and model input must have valid lengths.")
        for lengths in (self.scored, self.cutoffs):
            if lengths.ndim != 1 or len(lengths) == 0 or not np.all(np.diff(lengths) > 0):
                raise ValueError("Accuracy positions and PCA cutoffs must be increasing and unique.")
            if lengths[0] < 1 or lengths[-1] > n:
                raise ValueError("Accuracy positions and PCA cutoffs are one-based walk lengths.")
        n_nodes = len(self.board_ids)
        if self.adjacency.shape != (n_nodes, n_nodes) or self.nodes.min() < 0 or self.nodes.max() >= n_nodes:
            raise ValueError("The walk, graph, and node-to-token mapping do not agree.")

        windows = [np.arange(max(1, end - self.window + 1), end + 1) for end in self.cutoffs]
        self.pca_words = np.unique(np.concatenate(windows))
        evaluate_words = np.union1d(self.scored, self.pca_words)
        # Walk lengths count from one; model tensor positions count from zero.
        self.model_rows = prefix_length + evaluate_words - 1
        self.pca_indices = np.searchsorted(evaluate_words, self.pca_words)
        self.score_indices = np.searchsorted(evaluate_words, self.scored)

    def statistics(self, engine, final_states, probe_states):
        # Both inputs already contain just the relevant walk endpoints.
        valid_mass = np.empty(len(self.scored), dtype=np.float32)
        board = torch.as_tensor(self.board_ids, device=engine.device)
        for start in range(0, len(self.scored), engine.vocab_chunk):
            part = slice(start, start + engine.vocab_chunk)
            indices = torch.as_tensor(self.score_indices[part], device=engine.device)
            normalized = engine.base.norm(final_states.index_select(0, indices))
            logits = engine.model.lm_head(normalized).float()
            board_prob = (logits.index_select(-1, board) - torch.logsumexp(logits, dim=-1, keepdim=True)).exp()
            legal = self.adjacency[self.nodes[self.scored[part] - 1]]
            valid_mass[part] = (board_prob * torch.as_tensor(legal, device=engine.device)).sum(-1).cpu().numpy()

        # These CPU sums suffice to pool word means across walks. Release all raw activations afterwards.
        indices = torch.as_tensor(self.pca_indices, device=engine.device)
        residuals = probe_states.index_select(0, indices).float().cpu().numpy()
        shape = (len(self.cutoffs), len(self.board_ids))
        sums = np.zeros((*shape, residuals.shape[-1]), dtype=np.float32)
        counts = np.zeros(shape, dtype=np.int64)
        for i, end in enumerate(self.cutoffs):
            use = (self.pca_words <= end) & (self.pca_words > end - self.window)
            labels = self.nodes[self.pca_words[use] - 1]
            np.add.at(sums[i], labels, residuals[use])
            counts[i] = np.bincount(labels, minlength=len(self.board_ids))
        if not (np.isfinite(valid_mass).all() and np.isfinite(sums).all()):
            raise FloatingPointError("Non-finite scores or hidden states: the model overflows in this precision. Set "
                                     "PRECISION = 'float32', or use a GPU with bfloat16 (L4, A100), and run all "
                                     "cells again.")
        return valid_mass, sums, counts


class AblationEngine:
    """Decoder-only Llama, Qwen2/Qwen2.5, and Mistral in Transformers 4.56.2.

    A block reads earlier keys/values, zeros selected query-head outputs, then
    applies the model's ordinary output projection, residual connections, and MLP.
    Tensors represent one unpadded walk. H is query heads and K is key/value heads.
    """

    def __init__(self, model, query_chunk=64, vocab_chunk=64):
        self.model = model.eval()
        self.base = model.model
        self.layers = self.base.layers
        self.config = model.config
        if self.config.model_type not in ("llama", "qwen2", "mistral"):
            raise ValueError(f"Unsupported transformer architecture: {self.config.model_type}")
        self.n_heads = self.config.num_attention_heads
        self.n_kv = self.config.num_key_value_heads
        self.head_dim = self.layers[0].self_attn.head_dim
        self.device = self.base.embed_tokens.weight.device
        self.query_chunk, self.vocab_chunk = int(query_chunk), int(vocab_chunk)
        if min(self.query_chunk, self.vocab_chunk) < 1:
            raise ValueError("Choose positive attention and vocabulary chunks.")
        self.heads_per_key = self.n_heads // self.n_kv
        self.force_blocked = False   # set only by the engine check, to test the blocked attention path at any length
        self.windows = []
        for block in self.layers:
            window = getattr(block.self_attn, "sliding_window", None)
            if self.config.model_type == "mistral":
                window = self.config.sliding_window
            self.windows.append(window)

    def _head_map(self, heads):
        grouped = {}
        for layer, head in heads:
            layer, head = int(layer), int(head)
            if not (0 <= layer < len(self.layers) and 0 <= head < self.n_heads):
                raise ValueError(f"Unknown head at layer {layer}, head {head}.")
            grouped.setdefault(layer, []).append(head)
        if not grouped:
            raise ValueError("Each ablation group needs at least one head.")
        return grouped

    def _project(self, hidden, block, angles):
        """Project to Q/K/V and rotate Q/K at their true positions in the prompt."""
        attention = block.self_attn
        normed = block.input_layernorm(hidden)
        shape = (len(hidden), -1, self.head_dim)
        cos, sin = angles

        def rotate(x):
            left, right = x.chunk(2, dim=-1)
            return x * cos + torch.cat((-right, left), dim=-1) * sin

        query = rotate(attention.q_proj(normed).reshape(shape))
        key = rotate(attention.k_proj(normed).reshape(shape))
        value = attention.v_proj(normed).reshape(shape)
        return query, key.transpose(0, 1).contiguous(), value.transpose(0, 1).contiguous()

    def _finish(self, hidden, block, reads, zero):
        # Different query heads can share K/V. The cut is BEFORE the output projection
        # and affects only the listed query heads.
        if zero:
            reads[:, zero, :] = 0
        hidden = hidden + block.self_attn.o_proj(reads.flatten(1))
        # A gated MLP expands the hidden width several-fold. Bound its workspace.
        updated = torch.empty_like(hidden)
        for start in range(0, len(hidden), 512):
            rows = slice(start, start + 512)
            piece = hidden[rows]
            updated[rows] = piece + block.mlp(block.post_attention_layernorm(piece))
        return updated

    def _full_attention(self, query, key, value, scale, window):
        """Fused causal attention when possible, with bounded query blocks on older GPUs."""
        n = len(query)
        query = query.transpose(0, 1).unsqueeze(0)
        # Repeat only the current layer's KV heads. Persistent caches keep the compact K-head shape.
        key = key.repeat_interleave(self.heads_per_key, dim=0).unsqueeze(0)
        value = value.repeat_interleave(self.heads_per_key, dim=0).unsqueeze(0)
        global_attention = window is None or window >= n
        fast_device = self.device.type == "cpu" or (
            self.device.type == "cuda" and torch.cuda.get_device_capability(self.device)[0] >= 8
            and query.dtype in (torch.float16, torch.bfloat16)
        )
        if global_attention and not self.force_blocked and (fast_device or n <= 1024):
            result = F.scaled_dot_product_attention(query, key, value, is_causal=True, scale=scale)
            return result[0].transpose(0, 1).contiguous()
        result = torch.empty_like(query)
        positions = torch.arange(n, device=self.device)
        for start in range(0, n, self.query_chunk):
            end = min(start + self.query_chunk, n)
            left = 0 if global_attention else max(0, start - window + 1)
            current = positions[start:end, None]
            sources = positions[None, left:end]
            allowed = sources <= current
            if not global_attention:
                allowed = allowed & (sources > current - window)
            result[:, :, start:end] = F.scaled_dot_product_attention(
                query[:, :, start:end], key[:, :, left:end], value[:, :, left:end],
                attn_mask=allowed, scale=scale,
            )
        return result[0].transpose(0, 1).contiguous()

    def _endpoint_attention(self, query, private_key, private_value, bank, rows, scale, window):
        """Each row attends to strict history plus its OWN updated key and value.

        Example: to score word 20, positions 0..19 come from the shared cache
        (model positions counted from zero). Position 20 comes from that endpoint.
        Other replayed endpoints never enter this row's attention distribution.
        """
        keys, values = bank  # Each has shape [K, context, head_dim].
        model_positions = torch.as_tensor(rows, device=self.device)
        result = torch.empty_like(query)
        for start in range(0, len(rows), self.query_chunk):
            end = min(start + self.query_chunk, len(rows))
            part = slice(start, end)
            size = end - start
            right = int(rows[end - 1])   # The last endpoint also supplies its own current token.
            left = 0 if window is None else max(0, int(rows[start]) - window + 1)
            # Multiply all query heads sharing one KV head in a single matrix multiply.
            q = query[part].reshape(size, self.n_kv, self.heads_per_key, self.head_dim)
            q = q.permute(1, 0, 2, 3).reshape(self.n_kv, size * self.heads_per_key, self.head_dim)
            scores = torch.empty((self.n_kv, size, self.heads_per_key, right - left + 1),
                                 device=self.device, dtype=torch.float32)
            if right > left:
                hist = torch.bmm(q * scale, keys[:, left:right].transpose(1, 2))
                scores[..., :-1] = hist.reshape(self.n_kv, size, self.heads_per_key, right - left)
                sources = torch.arange(left, right, device=self.device)[None, :]
                current = model_positions[part, None]
                allowed = sources < current
                if window is not None:
                    allowed = allowed & (sources > current - window)
                scores[..., :-1].masked_fill_(~allowed[None, :, None, :], -torch.inf)
            own_query = q.reshape(self.n_kv, size, self.heads_per_key, self.head_dim)
            scores[..., -1] = (own_query.float() * private_key[:, part, None].float()).sum(-1) * scale
            weights = scores.softmax(dim=-1).to(values.dtype)
            read = weights[..., -1:] * private_value[:, part, None]
            if right > left:
                weights_hist = weights[..., :-1].reshape(self.n_kv, size * self.heads_per_key, right - left)
                old = torch.bmm(weights_hist, values[:, left:right]).reshape_as(read)
                read = read + old
            result[part] = read.permute(1, 0, 2, 3).reshape(size, self.n_heads, self.head_dim)
        return result

    def _prefill(self, hidden, first, angles, heads, probe, selected, *, cache=False, save_layers=()):
        """One causal pass. Save only shared KV, requested restart points, and the probe."""
        banks, snapshots = {}, {}
        for i in range(first, len(self.layers)):
            if i in save_layers:
                snapshots[i] = hidden
            if i == probe:
                geometry = hidden.index_select(0, selected)
            block = self.layers[i]
            q, k, v = self._project(hidden, block, angles)
            if cache:
                banks[i] = (k, v)
            reads = self._full_attention(q, k, v, block.self_attn.scaling, self.windows[i])
            hidden = self._finish(hidden, block, reads, heads.get(i))
            del q, k, v, reads
        return hidden.index_select(0, selected), geometry, banks, snapshots

    def _replay(self, hidden, first, angles, heads, banks, probe, rows):
        """Advance just the chosen independent endpoints against a previously computed history."""
        for i in range(first, len(self.layers)):
            if i == probe:
                geometry = hidden
            block = self.layers[i]
            q, own_k, own_v = self._project(hidden, block, angles)
            reads = self._endpoint_attention(
                q, own_k, own_v, banks[i], rows, block.self_attn.scaling, self.windows[i]
            )
            hidden = self._finish(hidden, block, reads, heads.get(i))
            del q, own_k, own_v, reads
        return hidden, geometry

    @torch.inference_mode()
    def measure_walk(self, token_ids, nodes, board_ids, adjacency, cutoffs, window, probe_layer,
                     groups, positions=None, pending=None):
        """Yield (group, scope, accuracy, vertex sums, vertex counts), one condition at a time.

        token_ids contains an optional tokenizer prefix, followed by one ID per graph node
        visited. positions and cutoffs count graph words starting at one. probe_layer
        denotes the residual entering a zero-based block. groups maps names to (layer, head) pairs.
        """
        ids = torch.as_tensor(token_ids, device=self.device, dtype=torch.long)
        if ids.ndim == 2 and ids.shape[0] == 1:
            ids = ids[0]
        if ids.ndim != 1:
            raise ValueError("Supply a single unpadded token sequence.")
        plan = WalkReadout(nodes, board_ids, adjacency, cutoffs, window, positions, len(ids) - len(nodes))
        expected = torch.as_tensor(plan.board_ids[plan.nodes], device=self.device)
        if not torch.equal(ids[-len(nodes):], expected):
            raise ValueError("The walk's token IDs do not match its word-to-node mapping.")
        if not 0 <= probe_layer < len(self.layers):
            raise ValueError("Choose a zero-based probe layer within this checkpoint.")
        if "intact" in groups:
            raise ValueError("'intact' is reserved for the baseline.")
        heads = {name: self._head_map(items) for name, items in groups.items()}
        conditions = {("intact", "intact")}
        conditions.update((name, scope) for name in groups for scope in ("final", "history", "everywhere"))
        pending = conditions if pending is None else set(pending)
        if not pending <= conditions:
            raise ValueError(f"Unknown intervention conditions: {pending - conditions}")
        if not pending:
            return
        initial = self.base.embed_tokens(ids)
        positions_tensor = torch.arange(len(ids), device=self.device)[None, :]
        angles = tuple(a[0, :, None] for a in self.base.rotary_emb(initial[None], positions_tensor))
        selected = torch.as_tensor(plan.model_rows, device=self.device)
        endpoint_angles = tuple(a.index_select(0, selected) for a in angles)
        need_final = any(scope == "final" for _, scope in pending)
        # Across groups, a late first ablation can reuse an unchanged stretch of blocks.
        # Cap this optional speed optimization so long contexts retain room for attention.
        first_edits = {min(min(heads[name]), probe_layer) for name, _ in pending if name != "intact"}
        budget = 256 * 1024**2
        if self.device.type == "cuda":
            budget = min(budget, torch.cuda.mem_get_info(self.device)[0] // 12)
        slots = budget // max(1, initial.numel() * initial.element_size())
        save_layers = sorted(first_edits - {0}, reverse=True)[:slots]
        starts = {0: initial}
        if need_final or ("intact", "intact") in pending:
            clean, geometry, banks, saved = self._prefill(
                initial, 0, angles, {}, probe_layer, selected, cache=need_final, save_layers=save_layers
            )
            starts.update(saved)
            if ("intact", "intact") in pending:
                yield ("intact", "intact", *plan.statistics(self, clean, geometry))
            del clean, geometry
            for name, edits in heads.items():
                if (name, "final") in pending:
                    first = max(i for i in starts if i <= min(min(edits), probe_layer))
                    branch = starts[first].index_select(0, selected)
                    final, geometry = self._replay(branch, first, endpoint_angles, edits, banks, probe_layer, plan.model_rows)
                    yield (name, "final", *plan.statistics(self, final, geometry))
                    del branch, final, geometry
            del banks  # Only one full multi-layer KV cache is resident at a time.
        for name, edits in heads.items():
            need_history = (name, "history") in pending
            if not need_history and (name, "everywhere") not in pending:
                continue
            first = max(i for i in starts if i <= min(min(edits), probe_layer))
            altered, geometry, banks, _ = self._prefill(
                starts[first], first, angles, edits, probe_layer, selected, cache=need_history
            )
            if (name, "everywhere") in pending:
                yield (name, "everywhere", *plan.statistics(self, altered, geometry))
            del altered, geometry
            if need_history:
                branch = starts[first].index_select(0, selected)
                final, geometry = self._replay(branch, first, endpoint_angles, {}, banks, probe_layer, plan.model_rows)
                yield (name, "history", *plan.statistics(self, final, geometry))
                del branch, final, geometry
            del banks

In [ ]:
#@title Load or reuse the selected checkpoint
if globals().get("_loaded_model_id") != (MODEL_ID, PRECISION) or "model" not in globals():
    # Drop references to the previous checkpoint before loading a different one.
    globals().pop("engine", None)
    globals().pop("model", None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tokenizer, model, revision, reference_1b_verified = load_checkpoint(MODEL_ID, PRECISION)
    _loaded_model_id = (MODEL_ID, PRECISION)
else:
    print("Reusing the loaded checkpoint.")
query_chunk = choose_query_chunk(QUERY_CHUNK)
engine = AblationEngine(model, query_chunk=query_chunk, vocab_chunk=LOGIT_ROWS)
probe_layer = choose_probe_layer(model.config.num_hidden_layers, PCA_LAYER)
print(f"{model.num_parameters():,} parameters in {model.dtype}. Residual entering block {probe_layer}. "
      f"Query block: {query_chunk} endpoints.")

In [ ]:
#@title Assign words to vertices and select attention heads
custom = [w.strip() for w in CUSTOM_WORDS.split(",")] if CUSTOM_WORDS.strip() else None
words, word_ids = choose_node_words(tokenizer, len(adjacency), custom, WORD_ASSIGNMENT_SEED)
word_ids = np.asarray(word_ids, dtype=np.int64)
prefix_ids = [] if tokenizer.bos_token_id is None else [int(tokenizer.bos_token_id)]
if NUMBER_OF_TOKENS + len(prefix_ids) > model.config.max_position_embeddings:
    raise ValueError("The requested walk plus its model prefix exceeds this checkpoint's context length.")
rankings, ranking_origin = choose_head_rankings(engine, MODEL_ID, revision, HEAD_SOURCE, reference_1b_verified)
groups, conditions = make_conditions(rankings, head_counts, RANDOM_CONTROL_DRAWS,
                                     model.config.num_hidden_layers, model.config.num_attention_heads,
                                     seed=113295, matched_draws=LAYER_MATCHED_DRAWS)
print(ranking_origin)
for k in head_counts:
    print(f"Top {k} of each family → {len(groups[f'both{k}'])} distinct heads in the joint ablation.")
random_sizes = sorted({(c["matches"], c["k"]) for c in conditions if c["family"] == "random"})
if random_sizes:
    print("Random controls from heads outside both top-32 lists: "
          + ", ".join(f"{k} heads (as many as the largest {'union' if kind == 'union' else 'single-family group'})"
                      for kind, k in random_sizes) + f"; {RANDOM_CONTROL_DRAWS} draw(s) each.")

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

edges = np.argwhere(np.triu(adjacency, k=1))
fig, ax = plt.subplots(figsize=(6, 5))
ax.add_collection(LineCollection(graph_xy[edges], colors="#a4b1b7", linewidths=1))
ax.scatter(*graph_xy.T, s=24, color="#167378")
if len(words) <= 40:
    for word, xy in zip(words, graph_xy):
        ax.annotate(word, xy, xytext=(3, 4), textcoords="offset points", fontsize=8)
ax.set_title(f"{graph_name} · {len(words)} single-token words")
ax.set_aspect("equal")
ax.margins(.14)
ax.axis("off")
plt.show()
plt.close(fig)

## 4. Check the engine, run and resume

**Engine check.** Before the run, the notebook compares the engine with ordinary Hugging Face forward passes on prefixes
of a walk, in which heads are zeroed by hooks on the attention output projection at the stated positions. It compares
the valid-neighbour probability and the probed hidden state, in all four conditions, twice: on a 12-word walk with the
engine's normal settings, and on a walk of up to 160 words with seven scored positions split into tiny blocks and with
the blocked attention path (the one used for long walks on a T4) switched on. The run stops if a difference exceeds
the tolerance: 2×10⁻⁴ in float32, and 0.04 in 16-bit precision, where fused and blocked attention add numbers in
different orders. The differences are saved in `native-check.json`.

**Resuming.** The notebook keeps one working file per condition, with the number of completed walks, the
valid-neighbour probability of every completed walk at every scored length, and pooled hidden-state sums and visit
counts per word and PCA window. After an interruption, each condition continues from its last completed walk if the
folder is still there; with `SAVE_TO_DRIVE` the folder survives the replacement of a Colab runtime. Changing the
experiment, the precision, the library versions or the measurement code starts a separate run folder. You can lower the
chunk sizes, or continue on another GPU with the same precision. Use one writer per run folder.

**Reading the code.** Start with `get_walk_inputs` and `run_walks` below, which turn one walk into a set of measured
conditions. Then follow `AblationEngine.measure_walk` in section 3: `_prefill` builds a shared history and `_replay`
recomputes the scored positions.

In [ ]:
#@title Independent reference implementation for a few short prefixes
"""Optional small, independent check using ordinary Hugging Face forward passes."""
from contextlib import ExitStack

import numpy as np
import torch


@torch.inference_mode()
def validate_engine(engine, token_ids, nodes, board_ids, adjacency, probe_layer, heads=None,
                    n_words=8, n_probes=3, query_chunk=None, force_blocked=False):
    """Compare all four scopes with ordinary forward passes on prefixes of a walk, with no cache reuse in the reference.
    n_words: walk length used; n_probes: scored positions, spread over the walk. query_chunk and force_blocked
    temporarily change the engine's block size and attention path, so that the check also covers scored positions
    split across blocks and the blocked attention used for long walks on older GPUs."""
    nodes = np.asarray(nodes)
    prefix = len(token_ids) - len(nodes)
    nodes = nodes[:n_words]
    ids = torch.as_tensor(token_ids[:prefix + len(nodes)], dtype=torch.long, device=engine.device)
    probes = np.unique(np.linspace(1, len(nodes), n_probes).round().astype(int))
    if heads is None:
        heads = [(0, 0), (len(engine.layers) - 1, engine.n_heads - 1)]
    grouped = engine._head_map(heads)
    fast = {}
    saved_settings = (engine.query_chunk, engine.force_blocked)
    try:
        if query_chunk is not None:
            engine.query_chunk = int(query_chunk)
        engine.force_blocked = bool(force_blocked)
        for _, scope, mass, sums, counts in engine.measure_walk(
            ids, nodes, board_ids, adjacency, probes, 1, probe_layer, {"check": heads}, positions=probes
        ):
            fast[scope] = (mass, sums[np.arange(len(probes)), nodes[probes - 1]])
    finally:
        engine.query_chunk, engine.force_blocked = saved_settings

    board = torch.as_tensor(board_ids, device=engine.device)
    errors = {}
    for scope, (observed_mass, observed_residual) in fast.items():
        expected_mass, expected_residual = [], []
        for words in probes:
            with ExitStack() as hooks:
                def capture(block, inputs):
                    expected_residual.append(inputs[0][0, -1].float().cpu().numpy())
                hooks.callback(engine.layers[probe_layer].register_forward_pre_hook(capture).remove)
                if scope != "intact":
                    sites = {"final": slice(-1, None), "history": slice(None, -1), "everywhere": slice(None)}[scope]
                    for layer, selected in grouped.items():
                        def remove_heads(projection, inputs, chosen=selected):
                            reads = inputs[0].clone().reshape(1, -1, engine.n_heads, engine.head_dim)
                            reads[:, sites, chosen, :] = 0
                            return (reads.reshape_as(inputs[0]),)
                        projection = engine.layers[layer].self_attn.o_proj
                        hooks.callback(projection.register_forward_pre_hook(remove_heads).remove)
                out = engine.model(ids[None, :prefix + words], use_cache=False, logits_to_keep=1)
                probabilities = out.logits[0, -1].float().softmax(-1)
                neighbors = torch.as_tensor(adjacency[nodes[words - 1]], device=engine.device, dtype=torch.bool)
                expected_mass.append(float(probabilities[board[neighbors]].sum()))
        reference = np.stack(expected_residual)
        relative = np.linalg.norm(observed_residual - reference) / max(np.linalg.norm(reference), 1e-12)
        errors[scope] = {"max_probability_error": float(np.abs(observed_mass - expected_mass).max()),
                         "relative_probe_error": float(relative)}
    # Fused and blocked attention use different accumulation orders at lower precisions.
    tolerance = 2e-4 if engine.base.embed_tokens.weight.dtype == torch.float32 else 0.04
    for scope, measures in errors.items():
        if any(not np.isfinite(value) or value > tolerance for value in measures.values()):
            raise AssertionError(f"Cached replay disagrees with native {scope} ablation: {measures}")
    return errors

In [ ]:
#@title Running sums and atomic checkpoints
"""Accumulate walk summaries, then plot the graph-tracing experiment on CPU."""

import csv
import json
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.collections import LineCollection
from matplotlib.ticker import NullLocator, PercentFormatter
import pandas as pd


def completed_walks(path):
    """Each condition has its own file. Walk indices start at zero."""
    if not Path(path).exists():
        return 0
    with np.load(path, allow_pickle=False) as saved:
        return int(saved["walks"].item())


def update_aggregate(path, walk_index, valid_mass, node_sums, node_counts):
    """Add one walk under the run's writer lock. A restart can retry its last walk."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    n = completed_walks(path)
    if walk_index < n:
        return False
    if walk_index != n:
        raise ValueError(f"{path.name} expects walk {n}, received {walk_index}.")

    mass = np.asarray(valid_mass, dtype=np.float32)
    sums = np.asarray(node_sums, dtype=np.float32)
    counts = np.asarray(node_counts)
    if mass.ndim != 1 or sums.ndim != 3 or counts.shape != sums.shape[:2]:
        raise ValueError("Expected scores[position], sums[window, node, feature], counts[window, node].")
    if not np.isfinite(mass).all() or mass.min() < -1e-5 or mass.max() > 1.0001:
        raise ValueError("A legal-neighbor probability is outside [0, 1].")
    if not np.isfinite(sums).all() or not np.issubdtype(counts.dtype, np.integer) or np.any(counts < 0):
        raise ValueError("Invalid PCA sums or node counts.")
    counts = counts.astype(np.int64, copy=False)
    by_walk = mass[None, :]                     # one row per completed walk, kept for the error bars

    if n:
        with np.load(path, allow_pickle=False) as saved:
            previous_mass = saved["mass_by_walk"]
            previous_sums = saved["node_sum"]
            if previous_mass.shape != (n, len(mass)) or previous_sums.shape != sums.shape:
                raise ValueError("The saved run has different positions, windows, nodes or hidden width.")
            by_walk = np.concatenate([previous_mass, by_walk])
            previous_sums += sums
            sums = previous_sums
            counts = saved["node_count"] + counts

    # Write the complete new checkpoint before replacing the old one. Keeping it
    # uncompressed makes repeated updates cheaper. The user export is compressed.
    tmp_name = None
    try:
        with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as tmp:
            tmp_name = tmp.name
            np.savez(tmp, walks=n + 1, mass_by_walk=by_walk, node_sum=sums, node_count=counts)
        os.replace(tmp_name, path)
    finally:
        if tmp_name is not None:
            Path(tmp_name).unlink(missing_ok=True)
    return True

In [ ]:
#@title Prepare each walk and dispatch the three ablation scopes
import inspect


def code_version():
    """Fingerprint of the code that produces the measurements, so that results from different code never mix."""
    functions = [build_graph, choose_node_words, walk_nodes, verify_walk_tokens, measure_head_rankings,
                 make_conditions, update_aggregate, WalkReadout.__init__, WalkReadout.statistics,
                 *[member for _, member in inspect.getmembers(AblationEngine, inspect.isfunction)]]
    try:
        source = "".join(inspect.getsource(function) for function in functions)
    except (OSError, TypeError):
        return "unhashed-2026-11"
    return hashlib.sha256(source.encode()).hexdigest()[:16]


CODE_VERSION = code_version()

def get_walk_inputs(sequence_index):
    nodes = walk_nodes(adjacency, NUMBER_OF_TOKENS, sequence_index, SEED)
    verify_walk_tokens(tokenizer, words, word_ids, nodes)
    ids = np.r_[np.asarray(prefix_ids, dtype=np.int64), word_ids[nodes]]
    return nodes, ids


def experiment_manifest():
    config = model.config
    return {
        "model": MODEL_ID, "model_revision": revision, "dtype": str(model.dtype),
        "transformers": transformers.__version__, "torch": str(torch.__version__),
        "gpu": torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU",
        "notebook_code": CODE_VERSION, "seed": SEED, "sequences": NUMBER_OF_SEQUENCES,
        "words_per_sequence": NUMBER_OF_TOKENS, "topology": TOPOLOGY, "graph": graph_name,
        "random_edge_probability": RANDOM_EDGE_PROBABILITY if TOPOLOGY == "random" else None,
        "edges": edges.tolist(), "words": list(words), "word_token_ids": word_ids.tolist(),
        "prefix_ids": prefix_ids, "accuracy_stride": ACCURACY_STRIDE,
        "pca_cutoffs": cutoffs, "pca_window": PCA_WINDOW, "pca_pre_block_zero_based": probe_layer,
        "rank_source": ranking_origin, "ranked_heads": rankings, "query_heads_per_layer": config.num_attention_heads,
        "group_heads": groups, "head_counts": head_counts, "conditions": conditions,
        "random_draw_seed": 113295, "layer_matched_draws": LAYER_MATCHED_DRAWS, "layer_matched_seed": [113295, 1],
        "word_assignment_seed": WORD_ASSIGNMENT_SEED,
        "actual_query_chunk": query_chunk, "logit_rows": LOGIT_ROWS,
    }


def run_walks(cache_dir):
    from tqdm.auto import tqdm
    # A group's full prefill and its endpoint replays may finish in different orders.
    # Persist each summary immediately so an interrupted run can pick up any missing scope.
    names = {("intact", "intact"): "intact"}
    for group in groups:
        for scope in ("everywhere", "history", "final"):
            names[(group, scope)] = f"{group}_{scope}"
    saved = {key: completed_walks(cache_dir / f"{name}.npz") for key, name in names.items()}
    with tqdm(total=NUMBER_OF_SEQUENCES * len(names), initial=sum(saved.values()), unit="condition") as progress:
        for walk_index in range(NUMBER_OF_SEQUENCES):
            pending = {key for key, n in saved.items() if n <= walk_index}
            if not pending:
                continue
            nodes, ids = get_walk_inputs(walk_index)
            iterator = engine.measure_walk(ids, nodes, word_ids, adjacency, cutoffs, PCA_WINDOW,
                                           probe_layer, groups, positions=score_at, pending=pending)
            for group, scope, mass, sums, counts in iterator:
                key = (group, scope)
                output = cache_dir / (names[key] + ".npz")
                update_aggregate(output, walk_index, mass, sums, counts)
                saved[key] = walk_index + 1
                progress.set_postfix_str(f"walk {walk_index + 1} · {names[key]}", refresh=False)
                progress.update(1)
                del mass, sums, counts

In [ ]:
#@title Set up the run folder and check projected storage
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive/graph-head-ablations")
else:
    root = Path("/content/graph-head-ablations")

manifest = experiment_manifest()
# Save hardware details with the execution settings. Precision and library versions stay
# in the experiment identity, so compatible GPUs can continue the same pooled sums.
execution = {key: manifest.pop(key) for key in ("gpu", "actual_query_chunk", "logit_rows")}
fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
run_dir = root / f"{MODEL_ID.split('/')[-1]}-{TOPOLOGY}-{fingerprint}"
cache_dir = run_dir / "running-sums"
export_dir = run_dir / "results"
cache_dir.mkdir(parents=True, exist_ok=True)
export_dir.mkdir(parents=True, exist_ok=True)
write_json(export_dir / "experiment.json", manifest)
history_file = export_dir / "execution-settings.json"
executions = json.loads(history_file.read_text(encoding="utf-8")) if history_file.exists() else []
if execution not in executions:
    write_json(history_file, executions + [execution])

d = model.config
element_bytes = model.get_input_embeddings().weight.element_size()
head_dim = getattr(d, "head_dim", None) or d.hidden_size // d.num_attention_heads
kv_bytes = 2 * d.num_hidden_layers * d.num_key_value_heads * head_dim * (NUMBER_OF_TOKENS + len(prefix_ids)) * element_bytes
sum_bytes = len(conditions) * len(cutoffs) * len(words) * d.hidden_size * 4
print(f"One full KV cache ≈ {kv_bytes / 2**30:.2f} GiB. Weights and temporary activations also use GPU memory.")
print(f"Working centroid sums ≈ {sum_bytes / 2**20:,.1f} MiB on disk. The download contains the fitted PCA coordinates.")
print("Run folder:", run_dir)

In [ ]:
#@title Cross-check the engine against ordinary Hugging Face forward passes
if RUN_NATIVE_CHECK:
    report = {}
    # 1. A 12-word walk, three scored positions, the engine's normal settings.
    check_nodes = walk_nodes(adjacency, 12, 0, SEED)
    check_ids = np.r_[np.asarray(prefix_ids, dtype=np.int64), word_ids[check_nodes]]
    report["short_walk"] = validate_engine(engine, check_ids, check_nodes, word_ids, adjacency, probe_layer,
                                           heads=rankings["induction"][:1] + rankings["previous"][:1])
    # 2. A longer walk with seven scored positions, blocks of three and the blocked attention path forced on.
    long_words = min(NUMBER_OF_TOKENS, 160)
    check_nodes = walk_nodes(adjacency, long_words, 1, SEED)
    check_ids = np.r_[np.asarray(prefix_ids, dtype=np.int64), word_ids[check_nodes]]
    report["longer_walk_blocked"] = validate_engine(
        engine, check_ids, check_nodes, word_ids, adjacency, probe_layer,
        heads=rankings["induction"][:4] + rankings["previous"][:4], n_words=long_words, n_probes=7,
        query_chunk=3, force_blocked=True)
    write_json(export_dir / "native-check.json", report)
    print("Engine checks passed:", json.dumps(report, indent=1))

In [ ]:
#@title Measure the walks and checkpoint pooled sums
from filelock import FileLock

try:
    with FileLock(str(run_dir / "writer.lock"), timeout=1):
        run_walks(cache_dir)
except torch.cuda.OutOfMemoryError as error:
    raise RuntimeError("GPU memory ran out. Lower QUERY_CHUNK and LOGIT_ROWS, restart the runtime, "
                       "and run all cells again. Completed conditions are reusable in the same run folder. "
                       "If the full KV cache or weights exceed the device capacity, shorten the walk or use a larger GPU.") from error
print("Complete. Every condition has the requested number of walks.")

## 5. Read the figures and download the results

**Valid-neighbour probability** is the probability, under the model's full-vocabulary softmax, of all neighbours of the
current word, including an immediate step back to the previous word. Every walk has equal weight at each length. The
curves are trailing 30-word means (each walk is smoothed, then averaged), so they start at word 30, where the first
full window ends; the shading is ±1 standard deviation across walks, as in Arditi's figures. The CSV files hold the
unsmoothed means at every word.

**Summary table** (`effects-summary.csv`). For each head family, head count and scope: the mean valid-neighbour
probability from `SUMMARY_FROM_WORD` to the end of the walk, its **drop** relative to the intact model on the same walks,
and a 95% bootstrap interval over walks. Where a matching control exists (layer-matched heads, or the random heads for
the largest k), the table also gives the control's drop and the **drop beyond the control**; the controls also have rows
of their own. `effects-table.png` shows the same table as an image: each cell gives the mean probability and the drop
with its interval, coloured by the size of the drop, with the crossover below. With few walks the intervals are rough;
use at least 16 walks for numbers you report.

**Crossover** (`crossover-summary.csv`). For each k:
(drop with the previous-token heads removed in the history − drop with them removed at the final word)
− (the same difference for the induction heads). A positive value is the pattern the textbook circuit predicts:
previous-token heads matter relatively more in the history, induction heads at the final word.

**PCA maps.** For each condition and window, the word centroids (pooled over walks) are centred with equal weight and
projected onto a fresh pair of principal components. Lines join graph neighbours. Axes can flip or rotate between
panels, and early panels may contain fewer words; check the coverage and explained variance in each title.

In the final-word-only and history-only maps, *every* word of the window is treated as the final word of its own
prefix (section 3), not only the last word of the walk. The final-word-only map shows each word's hidden state when the
heads are switched off at that word but its past is computed normally; the history-only map shows each word's state
when its past was computed without the heads but the heads work at the word itself. So these maps differ from the intact
and everywhere maps at every word in the window. (Switching the heads off at the walk's last word alone would change one
word out of the window, and the map would look like the intact one.) Only heads in blocks below the probed layer
(`PCA_LAYER`) can change a map: the hidden state entering a block depends only on the blocks before it.

**Graph overlap** compares the PC1–PC2 plane with the plane of the two smoothest non-constant eigenvectors of the graph
Laplacian (1 = the same plane). It is reported only when all words have appeared, the PCA plane has rank two, and the
graph defines a unique two-dimensional plane.

The ZIP contains the figures, a PDF with every PCA panel, the CSV tables, a NumPy archive with the per-walk
valid-neighbour probability, PCA coordinates, explained variances and counts, a README describing every file, and JSON
records of the model revision, graph, words, head lists, seeds and settings. Raw hidden states, logits and the working
files stay out of the download.

In [ ]:
#@title Fit the PCA maps and draw the figures
def word_pca(sums, counts):
    """Fit two PCs to observed word means. Every observed word has equal weight."""
    seen = counts > 0
    xy = np.full((len(counts), 2), np.nan)
    variance = np.full(2, np.nan)
    if not seen.any():
        return xy, variance
    means = sums[seen].astype(np.float64) / counts[seen, None]
    means -= means.mean(axis=0)
    # There are usually many fewer words than residual features. Diagonalizing
    # this small word-by-word matrix gives the same scores as PCA via SVD.
    gram = means @ means.T
    values, vectors = np.linalg.eigh(gram)
    n_components = min(2, len(values))
    values = values[::-1][:n_components].clip(min=0)
    scores = vectors[:, ::-1][:, :n_components] * np.sqrt(values)
    # PC signs are arbitrary. Pick repeatable signs for plotting.
    for j in range(n_components):
        if scores[np.argmax(np.abs(scores[:, j])), j] < 0:
            scores[:, j] *= -1
    xy[seen] = 0
    xy[seen, :n_components] = scores
    if gram.trace() > 0:
        variance[:n_components] = values / gram.trace()
    return xy, variance


def spectral_alignment(adjacency, layouts):
    """Compare the two-dimensional PCA plane with the graph's two smoothest modes."""
    scores = np.full(layouts.shape[:-2], np.nan)
    n = len(adjacency)
    if n < 3:
        return scores
    laplacian = np.diag(adjacency.sum(axis=1)) - adjacency.astype(float)
    eigenvalues, modes = np.linalg.eigh(laplacian)
    tolerance = 32 * n * np.finfo(float).eps * max(1.0, eigenvalues[-1])
    # The first mode is constant. A tie across the outer boundary means that
    # there is no unique two-dimensional target. A tie within the pair is fine.
    if eigenvalues[1] <= tolerance or (n > 3 and eigenvalues[3] - eigenvalues[2] <= tolerance):
        return scores
    graph_plane = modes[:, 1:3]
    for idx in np.ndindex(scores.shape):
        xy = layouts[idx]
        if not np.isfinite(xy).all():
            continue
        plane, scales, _ = np.linalg.svd(xy - xy.mean(axis=0), full_matrices=False)
        if scales[1] > scales[0] * np.sqrt(n * np.finfo(float).eps):
            overlap = np.square(graph_plane.T @ plane[:, :2]).sum() / 2
            scores[idx] = np.clip(overlap, 0, 1)
    return scores


def read_summaries(cache_dir, words, positions, cutoffs, conditions, expected_walks, window_size):
    """Read one condition at a time so full-width residuals need not coexist. Returns the valid-neighbour probability
    of every walk [condition, walk, position], PCA coordinates, explained variances, word counts and walk count."""
    masses, coordinates, variances = [], [], []
    walk_total = None
    shared_counts = None
    for cond in conditions:
        path = cache_dir / (cond["name"] + ".npz")
        with np.load(path, allow_pickle=False) as saved:
            n = int(saved["walks"].item())
            if n < 1 or (expected_walks is not None and n != expected_walks):
                raise ValueError(f"{path.name}: expected {expected_walks} completed walks; found {n}.")
            counts = saved["node_count"]
            sums = saved["node_sum"]
            mass = saved["mass_by_walk"]
        if walk_total is None:
            walk_total = n
            shared_counts = counts
        if n != walk_total or not np.array_equal(counts, shared_counts):
            raise ValueError(f"{path.name} used different walks or PCA positions.")
        if counts.shape != (len(cutoffs), len(words)) or mass.shape != (n, len(positions)):
            raise ValueError(f"Shape mismatch in {path.name}. Check the experiment settings.")
        if not np.array_equal(counts.sum(axis=1), n * np.minimum(cutoffs, window_size)):
            raise ValueError(f"{path.name} does not cover all words in the stated PCA windows.")
        masses.append(mass)
        fitted = [word_pca(s, c) for s, c in zip(sums, counts)]
        coordinates.append([item[0] for item in fitted])
        variances.append([item[1] for item in fitted])
        del sums
    return np.asarray(masses), np.asarray(coordinates), np.asarray(variances), shared_counts, walk_total


FAMILY_LABEL = {"induction": "Induction heads", "previous": "Previous-token heads",
                "both": "Union of both head sets", "random": "Random controls",
                "induction_matched": "Layer-matched control (induction)",
                "previous_matched": "Layer-matched control (previous-token)",
                "both_matched": "Layer-matched control (union)"}
SCOPE_LABEL = {"everywhere": "Everywhere", "history": "History only", "final": "Final word only"}
SCOPE_COLOR = {"everywhere": "#963f64", "history": "#c1722b", "final": "#007e87"}


def _ordered_fields(conditions, field, labels):
    available = {c[field] for c in conditions}
    return [key for key in labels if key in available]


def _matching(conditions, family, scope, k=None):
    return [i for i, c in enumerate(conditions)
            if c["family"] == family and c["scope"] == scope and (k is None or c["k"] == k)]


def _random_ids(conditions, kind, scope):
    """Random controls sized like the largest single-family group (kind "single") or the largest union ("union")."""
    return [i for i, c in enumerate(conditions)
            if c["family"] == "random" and c.get("matches") == kind and c["scope"] == scope]


def _smooth(positions, scores, smooth):
    """Trailing mean over a word-length window, even if accuracy was sampled sparsely; each row is one walk.
    Returns the positions from the first full window on, with the smoothed rows; a walk shorter than the window is
    returned unsmoothed."""
    scores = np.atleast_2d(np.asarray(scores, dtype=np.float64))
    # O(number of sampled positions), with no filling-in of missing scores.
    start = np.searchsorted(positions, positions - smooth + 1)
    cumulative = np.concatenate([np.zeros((len(scores), 1)), np.cumsum(scores, axis=1)], axis=1)
    stop = np.arange(1, scores.shape[1] + 1)
    rolling = (cumulative[:, stop] - cumulative[:, start]) / (stop - start)
    shown = positions >= smooth
    if not shown.any():
        return positions, scores
    return positions[shown], rolling[:, shown]


def _line(ax, positions, walks, smooth, band=False, **kwargs):
    """Mean over walks of the smoothed curves; optionally shade ±1 standard deviation across walks."""
    shown, curves = _smooth(positions, walks, smooth)
    mean = curves.mean(axis=0)
    if len(shown) == 1:
        kwargs["marker"] = "o"
    line, = ax.plot(shown, mean, **kwargs)
    if band and len(curves) > 1:
        spread = curves.std(axis=0, ddof=1)
        ax.fill_between(shown, mean - spread, mean + spread, color=line.get_color(), alpha=.14, linewidth=0)


def _accuracy_axes(ax, positions, first=None):
    ax.set_ylim(0, 1.02)
    ax.set_xlim((positions[0] if first is None else first) - .5, positions[-1] + .5)
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.grid(axis="y", color="#dae1e3", linewidth=.7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("Context length in words")


def _cut_label(family, k):
    if family.endswith("_matched"):
        return "Matched to " + _cut_label(family[:-len("_matched")], k).lower()
    return f"Top {k} of each" if family == "both" else f"Top {k}"


def _accuracy_figures(folder, positions, mass, conditions, context, smooth):
    families = [f for f in _ordered_fields(conditions, "family", FAMILY_LABEL) if f != "random"]
    scopes = _ordered_fields(conditions, "scope", SCOPE_LABEL)
    intact = next(i for i, c in enumerate(conditions) if c["family"] == "intact")
    counts = sorted({c["k"] for c in conditions if c["family"] in families})
    shown = positions[positions >= smooth]
    first = int(shown[0]) if len(shown) else int(positions[0])   # the curves start at the first full window
    smoothing = (f"Trailing {smooth}-word means from word {first}" if len(shown)
                 else "Unsmoothed: the walks are shorter than the smoothing window")
    palette = plt.get_cmap("viridis")(np.linspace(.08, .78, len(counts)))
    cut_colors = dict(zip(counts, palette))
    accuracy_pdf = folder / "accuracy-by-head-count.pdf"
    preview = folder / "accuracy-by-head-count.png"
    pages = [counts[i:i+7] for i in range(0, len(counts), 7)] or [[]]
    with PdfPages(accuracy_pdf) as pdf:
        for page, selected in enumerate(pages):
            fig, axes = plt.subplots(max(1, len(families)), max(1, len(scopes)), squeeze=False,
                                     figsize=(4.3 * max(1, len(scopes)), 2.75 * max(1, len(families)) + 1.45))
            for row, family in enumerate(families or ["intact"]):
                for col, scope in enumerate(scopes or ["everywhere"]):
                    ax = axes[row, col]
                    _line(ax, positions, mass[intact], smooth, band=True, color="#203847", lw=2.0, label="Intact")
                    for k in selected:
                        idx = _matching(conditions, family, scope, k)
                        if idx:
                            _line(ax, positions, mass[idx].mean(axis=0), smooth,
                                  color=cut_colors[k], lw=1.45, label=f"k = {k}")
                    if family in ("induction", "previous", "both"):
                        idx = _random_ids(conditions, "union" if family == "both" else "single", scope)
                        if idx:
                            _line(ax, positions, mass[idx].mean(axis=0), smooth, color="#757177",
                                  lw=1.45, ls="--", label="Random heads, as many as the largest group")
                    ax.set_title(f"{FAMILY_LABEL.get(family, 'Intact')} · {SCOPE_LABEL[scope]}", fontsize=10)
                    _accuracy_axes(ax, positions, first)
                    if col == 0:
                        ax.set_ylabel("Valid-neighbour probability")
            handles, names = axes[0, 0].get_legend_handles_labels()
            fig.legend(handles, names, loc="lower center", ncol=len(names), frameon=False)
            height = fig.get_size_inches()[1]
            fig.suptitle(f"Valid-neighbour probability by head count\n{context}\n{smoothing} · "
                         "shading: ±1 s.d. of the intact model across walks", fontsize=12, y=1 - 0.15 / height)
            fig.tight_layout(rect=(0, 0.5 / height, 1, 1))     # tight_layout also makes room for the title
            pdf.savefig(fig)
            if page == 0:
                fig.savefig(preview, dpi=155)
            plt.close(fig)

    # A fixed head count per panel makes the contrast between scopes easy to see.
    panels = [(family, max(c["k"] for c in conditions if c["family"] == family), None, None) for family in families]
    for kind in ("single", "union"):
        found = [c for c in conditions if c["family"] == "random" and c.get("matches") == kind]
        if found:
            panels.append(("random", found[0]["k"], kind, found[0]["matched_k"]))
    if not panels:
        panels = [("intact", 0, None, None)]
    cols = min(len(panels), 2)
    rows = (len(panels) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, squeeze=False, figsize=(5.5 * cols, 3.3 * rows + 1.0))
    for ax, (family, k, kind, matched) in zip(axes.flat, panels):
        _line(ax, positions, mass[intact], smooth, band=True, color="#203847", lw=2, ls="--", label="Intact")
        for scope in scopes:
            idx = (_random_ids(conditions, kind, scope) if family == "random"
                   else _matching(conditions, family, scope, k))
            if idx:
                _line(ax, positions, mass[idx].mean(axis=0), smooth, band=True, color=SCOPE_COLOR[scope],
                      lw=1.7, label=SCOPE_LABEL[scope])
        if family == "random":
            label = f"{k} heads, as many as " + (f"both top {matched} together" if kind == "union" else f"top {matched}")
        else:
            label = _cut_label(family, k)
        ax.set_title(f"{FAMILY_LABEL.get(family, 'Intact')} · {label}" if k else "Intact", fontsize=10)
        ax.set_ylabel("Valid-neighbour probability")
        _accuracy_axes(ax, positions, first)
    for ax in list(axes.flat)[len(panels):]:
        ax.set_visible(False)
    handles, names = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, names, loc="lower center", ncol=len(handles), frameon=False)
    height = fig.get_size_inches()[1]
    fig.suptitle(f"Where the heads were ablated\n{context}\n{smoothing} · shading: ±1 s.d. across walks", fontsize=12,
                 y=1 - 0.15 / height)
    fig.tight_layout(rect=(0, 0.5 / height, 1, 1))     # tight_layout also makes room for the title
    scope_png = folder / "accuracy-by-scope.png"
    fig.savefig(scope_png, dpi=155)
    plt.close(fig)
    return accuracy_pdf, preview, scope_png


def _place_names(ax, points, labels):
    """Try nearby label positions, favoring empty space and short leader lines."""
    directions = np.array([(x, y) for x in (-1, 0, 1) for y in (-1, 0, 1) if x or y])
    directions = np.tile(directions, (4, 1))
    gaps = np.repeat([.014, .045, .09, .15], 8)
    inches = ax.figure.get_size_inches() * ax.get_position().size
    font = 6
    unit = font / (72 * min(inches))
    # Work in panel coordinates between zero and one. All boxes are x-low, y-low, x-high, y-high.
    points = np.asarray(points)
    obstacles = np.column_stack((points - .013, points + .013))
    for point, label in zip(points, labels):
        half = np.array([len(label) * .33 + .22, .62]) * unit
        centers = point + directions * (half + gaps[:, None])
        boxes = np.column_stack((centers - half, centers + half))
        lower = np.maximum(boxes[:, None, :2], obstacles[:, :2])
        upper = np.minimum(boxes[:, None, 2:], obstacles[:, 2:])
        collision = np.maximum(0, upper - lower).prod(axis=2).sum(axis=1)
        outside = np.maximum(0, -boxes[:, :2]).sum(1) + np.maximum(0, boxes[:, 2:] - 1).sum(1)
        best = np.argmin(2000 * collision + 10 * outside + gaps)
        obstacles = np.vstack((obstacles, boxes[best]))
        leader = dict(arrowstyle="-", lw=.45, color="#69808a") if gaps[best] > .035 else None
        ax.annotate(label, point, xycoords="axes fraction", xytext=centers[best],
                    textcoords="axes fraction", ha="center", va="center", fontsize=font,
                    bbox=dict(facecolor="white", alpha=.85, edgecolor="none", pad=.15),
                    arrowprops=leader)


def _draw_map(ax, xy, edges, words, colors, title):
    seen = np.isfinite(xy).all(axis=1)
    visible_edges = edges[seen[edges].all(axis=1)]
    if len(visible_edges):
        ax.add_collection(LineCollection(xy[visible_edges], colors="#a4b0b6", linewidths=.65))
    if seen.any():
        ax.scatter(xy[seen, 0], xy[seen, 1], c=colors[seen], s=17, zorder=3,
                   edgecolors="white", linewidths=.35)
        points = xy[seen]
        span = max(float(np.ptp(points, axis=0).max()) * 1.45, .01)
        low = (points.max(0) + points.min(0) - span) / 2
        ax.set(xlim=(low[0], low[0]+span), ylim=(low[1], low[1]+span), aspect="equal")
        if len(words) <= 32:
            labels = [words[i] if len(words[i]) <= 15 else str(i) for i in np.flatnonzero(seen)]
            _place_names(ax, (points - low) / span, labels)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, fontsize=9, pad=5)


def _pca_figures(folder, words, graph_xy, edges, layouts, variance, overlap, node_count, conditions,
                 cutoffs, window_size, context):
    # One color per node in every panel, including the original graph.
    colors = plt.get_cmap("turbo")(np.linspace(.08, .90, len(words)))
    intact = next(i for i, c in enumerate(conditions) if c["family"] == "intact")
    groups = {}
    for i, c in enumerate(conditions):
        if c["family"] != "intact":
            groups.setdefault((c["family"], c["scope"]), []).append(i)
    if not groups:
        groups[("intact", "intact")] = []
    for ids in groups.values():
        ids.sort(key=lambda i: (conditions[i]["k"], conditions[i]["name"]))
    sorted_groups = sorted(groups, key=lambda g: (
        list(FAMILY_LABEL).index(g[0]) if g[0] in FAMILY_LABEL else -1,
        list(SCOPE_LABEL).index(g[1]) if g[1] in SCOPE_LABEL else -1))
    max_edges = 384
    if len(edges) > max_edges:
        select = np.linspace(0, len(edges)-1, max_edges, dtype=int)
        edges = edges[select]
        edge_note = f"Drawing {max_edges} graph edges. All edges contribute to accuracy."
    else:
        edge_note = "Lines connect true graph neighbors."

    def draw_centroids(ax, idx, win, title):
        explained = variance[idx, win]
        suffix = f"\nPC1+2 {np.nansum(explained):.0%}" if np.isfinite(explained).any() else ""
        if np.isfinite(overlap[idx, win]):
            suffix += f" · overlap {overlap[idx, win]:.2f}"
        _draw_map(ax, layouts[idx, win], edges, words, colors, title + suffix)

    def names_for(ids):
        titles = []
        for idx in ids:
            c = conditions[idx]
            if c["family"] == "random":
                # A distinct projection for each random draw; never average PCA axes.
                draws = _random_ids(conditions, c.get("matches"), c["scope"])
                size = " (union size)" if c.get("matches") == "union" else ""
                titles.append(f"Random {c['k']}{size} · draw {draws.index(idx)+1}")
            elif c["family"].endswith("_matched"):
                draws = _matching(conditions, c["family"], c["scope"], c["k"])
                titles.append(f"{_cut_label(c['family'], c['k'])} · draw {draws.index(idx)+1}")
            else:
                titles.append(_cut_label(c["family"], c["k"]))
        return titles

    pca_pdf = folder / "pca-all-conditions.pdf"
    with PdfPages(pca_pdf) as pdf:
        for w, end in enumerate(cutoffs):
            start = max(1, end - window_size + 1)
            covered = np.count_nonzero(node_count[w])
            for family, scope in sorted_groups:
                ids = groups[family, scope]
                chunks = [ids[j:j+7] for j in range(0, len(ids), 7)] or [[]]
                for subset in chunks:
                    n_panels = len(subset) + 2
                    cols = min(3, n_panels)
                    rows = (n_panels + cols - 1) // cols
                    fig, axes = plt.subplots(rows, cols, squeeze=False,
                                             figsize=(3.9 * cols, 3.15 * rows + 1.1))
                    fig.subplots_adjust(top=.80 if rows == 1 else .88, bottom=.09,
                                        left=.035, right=.965, hspace=.34, wspace=.16)
                    axes = list(axes.flat)
                    _draw_map(axes[0], graph_xy, edges, words, colors, "Reference graph")
                    draw_centroids(axes[1], intact, w, "Intact")
                    for ax, idx, title in zip(axes[2:], subset, names_for(subset)):
                        draw_centroids(ax, idx, w, title)
                    for ax in axes[n_panels:]:
                        ax.set_visible(False)
                    heading = FAMILY_LABEL.get(family, "Intact")
                    if scope in SCOPE_LABEL:
                        heading += " · " + SCOPE_LABEL[scope]
                    fig.suptitle(f"{heading}\n{context} · Words {start:,}–{end:,} · {covered}/{len(words)} words observed",
                                 fontsize=12)
                    fig.text(.035, .018, "PC1 is horizontal and PC2 vertical. Each centroid panel has its own PCA fit. " + edge_note,
                             fontsize=8)
                    pdf.savefig(fig)
                    plt.close(fig)

    # One small summary image for the notebook preview. All counts, scopes,
    # random draws and cutoff windows appear in the PDF above.
    end_index = len(cutoffs) - 1
    scopes = _ordered_fields(conditions, "scope", SCOPE_LABEL) or ["intact"]
    families = _ordered_fields(conditions, "family", FAMILY_LABEL)
    cols = 1 + len(families)
    fig, axes = plt.subplots(len(scopes), cols, squeeze=False,
                             figsize=(3.1 * cols, 2.75 * len(scopes) + 1.15))
    fig.subplots_adjust(top=.83 if len(scopes) == 1 else .88, bottom=.08,
                        left=.06, right=.97, hspace=.36, wspace=.20)
    for row, scope in enumerate(scopes):
        draw_centroids(axes[row, 0], intact, end_index, "Intact")
        for col, family in enumerate(families, start=1):
            ids = groups.get((family, scope), [])
            if not ids:
                axes[row, col].set_visible(False)
                continue
            highest = max(conditions[i]["k"] for i in ids)
            idx = next(i for i in ids if conditions[i]["k"] == highest)
            short_family = {"induction": "Induction", "previous": "Previous-token", "both": "Union"}
            if family == "random":
                size = " (union size)" if conditions[idx].get("matches") == "union" else ""
                spec = f"Random {highest}{size} · draw 1"
            else:
                spec = f"{short_family.get(family, 'Control')} · {_cut_label(family, highest)}"
            draw_centroids(axes[row, col], idx, end_index, spec)
        axes[row, 0].set_ylabel(SCOPE_LABEL.get(scope, "Intact"), weight="bold", fontsize=9)
    tail = min(window_size, int(cutoffs[-1]))
    observed = np.count_nonzero(node_count[-1])
    fig.suptitle(f"Word geometry after {cutoffs[-1]:,} words\n{context} · Last {tail} words per walk · {observed}/{len(words)} nodes observed",
                 fontsize=13)
    fig.text(.035, .015, "PC1 is horizontal and PC2 vertical. " + edge_note, fontsize=8)
    preview = folder / "pca-final-window.png"
    fig.savefig(preview, dpi=155)
    plt.close(fig)
    return pca_pdf, preview


# Row order of the summary: ablated head groups, then their layer-matched controls, then the random controls.
SUMMARY_ORDER = ("induction", "previous", "both", "induction_matched", "previous_matched", "both_matched",
                 "random", "random_union")


def summarize_effects(mass_by_walk, positions, conditions, from_word, n_boot=10000, seed=0):
    """Drop in valid-neighbour probability, from word `from_word` to the end of the walk, for every head family, count
    and scope. Walks are resampled together (a paired bootstrap), so every condition stays paired with the intact model
    on the same walks. Draws of the same control are averaged within each walk first."""
    window = positions >= from_word
    per_walk = mass_by_walk[:, :, window].mean(axis=2)                    # [condition, walk]
    n_walks = per_walk.shape[1]
    intact = per_walk[next(i for i, c in enumerate(conditions) if c["family"] == "intact")]
    resample = np.random.default_rng(seed).integers(n_walks, size=(n_boot, n_walks))

    def interval(values):
        if n_walks < 2:
            return np.nan, np.nan
        low, high = np.percentile(values[resample].mean(axis=1), [2.5, 97.5])
        return low, high

    cells, matched_k = {}, {}               # (family, k, scope) -> valid-neighbour probability of each walk
    for c in conditions:
        if c["family"] == "intact":
            continue
        family = "random_union" if c.get("matches") == "union" else c["family"]
        if (family, c["k"], c["scope"]) not in cells:
            ids = [i for i, d in enumerate(conditions) if (d["family"], d["k"], d["scope"], d.get("matches"))
                   == (c["family"], c["k"], c["scope"], c.get("matches"))]
            cells[family, c["k"], c["scope"]] = per_walk[ids].mean(axis=0)
        if c["family"] == "random":
            matched_k[family, c["k"]] = c.get("matched_k")

    def control_for(family, k, scope):
        if family in ("induction", "previous", "both"):
            if (f"{family}_matched", k, scope) in cells:
                return "layer-matched", cells[f"{family}_matched", k, scope]
            kind = "random_union" if family == "both" else "random"
            for (other, size, other_scope), values in cells.items():
                if other == kind and other_scope == scope and matched_k.get((other, size)) == k:
                    return f"{size} random heads", values
        return None, None

    family_order = {name: i for i, name in enumerate(SUMMARY_ORDER)}
    scope_order = {name: i for i, name in enumerate(SCOPE_LABEL)}
    rows = []
    for family, k, scope in sorted(cells, key=lambda key: (family_order.get(key[0], 99), key[1], scope_order[key[2]])):
        drop = intact - cells[family, k, scope]
        low, high = interval(drop)
        row = {"family": family, "k": k, "matched_k": matched_k.get((family, k)), "scope": scope, "walks": n_walks,
               "from_word": from_word, "intact": intact.mean(), "accuracy": cells[family, k, scope].mean(),
               "drop": drop.mean(), "drop_low": low, "drop_high": high}
        name, control = control_for(family, k, scope)
        if control is not None:
            beyond = drop - (intact - control)
            low, high = interval(beyond)
            row.update(control=name, control_drop=(intact - control).mean(), drop_beyond_control=beyond.mean(),
                       beyond_low=low, beyond_high=high)
        rows.append(row)
    effects = pd.DataFrame(rows)
    effects["matched_k"] = effects["matched_k"].astype("Int64")     # blank except for the random controls

    crossover_rows = []
    for k in sorted({key[1] for key in cells if key[0] in ("induction", "previous")}):
        needed = [(family, k, scope) for family in ("previous", "induction") for scope in ("history", "final")]
        if not all(key in cells for key in needed):
            continue
        drops = {key: intact - cells[key] for key in needed}
        previous_gap = drops["previous", k, "history"] - drops["previous", k, "final"]
        induction_gap = drops["induction", k, "history"] - drops["induction", k, "final"]
        crossover = previous_gap - induction_gap
        low, high = interval(crossover)
        crossover_rows.append({"k": k, "walks": n_walks, "from_word": from_word,
                               "previous_history_minus_final": previous_gap.mean(),
                               "induction_history_minus_final": induction_gap.mean(),
                               "crossover": crossover.mean(), "crossover_low": low, "crossover_high": high})
    return effects, (pd.DataFrame(crossover_rows) if crossover_rows else None)


def _effects_figure(folder, effects, context, from_word, last_word):
    """Drop relative to intact against k, one panel per head family and one line per scope."""
    families = [f for f in ("induction", "previous", "both") if f in set(effects["family"])]
    if not families:
        return None
    ks = sorted(set(effects.loc[effects.family.isin(families), "k"]))
    fig, axes = plt.subplots(1, len(families), squeeze=False, figsize=(4.6 * len(families), 3.9), sharey=True)
    for ax, family in zip(axes[0], families):
        for scope in SCOPE_LABEL:
            part = effects[(effects.family == family) & (effects.scope == scope)].sort_values("k")
            if len(part):
                error = np.vstack([part["drop"] - part["drop_low"], part["drop_high"] - part["drop"]]).clip(min=0)
                ax.errorbar(part["k"], 100 * part["drop"], yerr=100 * np.nan_to_num(error), marker="o", ms=4, lw=1.5,
                            capsize=2, color=SCOPE_COLOR[scope], label=SCOPE_LABEL[scope])
            matched = effects[(effects.family == f"{family}_matched") & (effects.scope == scope)].sort_values("k")
            if len(matched):
                ax.plot(matched["k"], 100 * matched["drop"], ls="--", lw=1, color=SCOPE_COLOR[scope],
                        label=f"{SCOPE_LABEL[scope]}, layer-matched control")
            kind = "random_union" if family == "both" else "random"
            random = effects[(effects.family == kind) & (effects.scope == scope)]
            if len(random):
                ax.plot(random["matched_k"].astype(float), 100 * random["drop"], marker="s", ls="none", mfc="white",
                        color=SCOPE_COLOR[scope], label=f"{SCOPE_LABEL[scope]}, random heads, as many as the largest group")
        ax.axhline(0, color="#8a9599", lw=.8)
        ax.set_xscale("log", base=2)
        ax.set_xticks(ks)
        ax.set_xticklabels([str(k) for k in ks])
        ax.xaxis.set_minor_locator(NullLocator())
        ax.set_xlabel("Heads removed per family (k)")
        ax.set_title(FAMILY_LABEL[family], fontsize=10)
        ax.grid(axis="y", color="#dae1e3", linewidth=.7)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0, 0].set_ylabel("Drop in valid-neighbour probability\n(percentage points)")
    labels = {}
    for ax in axes[0]:
        for handle, name in zip(*ax.get_legend_handles_labels()):
            labels.setdefault(name, handle)
    scope_names = list(SCOPE_LABEL.values())

    def legend_key(name):
        kind = 0 if "," not in name else 1 if "layer-matched" in name else 2
        scope = next((i for i, s in enumerate(scope_names) if name.startswith(s)), 9)
        return kind, scope, name

    order = sorted(labels, key=legend_key)
    legend_rows = (len(order) + 2) // 3
    height = 3.15 + 0.26 * legend_rows + 0.9
    fig.set_size_inches(4.6 * len(families), height)
    fig.legend([labels[name] for name in order], order, loc="lower center", ncol=min(3, len(order)), frameon=False,
               fontsize=8)
    fig.suptitle(f"Drop relative to the intact model, words {from_word:,}–{last_word:,}\n{context} · "
                 "error bars: 95% bootstrap intervals over walks", fontsize=12, y=1 - 0.15 / height)
    fig.tight_layout(rect=(0, (0.26 * legend_rows + 0.15) / height, 1, 1))
    path = folder / "effects-by-head-count.png"
    fig.savefig(path, dpi=155)
    plt.close(fig)
    return path


def _signed(value):
    """A fraction as signed percentage points with one decimal; rounding never prints -0.0."""
    return f"{round(100 * value, 1) + 0.0:+.1f}"


def _effects_table_figure(folder, effects, crossover, context, from_word, last_word, max_counts=8):
    """Both summary tables as one image. Each cell gives the mean valid-neighbour probability over the summary words
    and, below it, the drop relative to the intact model in percentage points with its 95% interval over walks. The
    colour shows the drop: red below the intact model, blue above it, white near zero."""
    if effects is None or effects.empty:
        return None
    scopes = [s for s in SCOPE_LABEL if s in set(effects["scope"])]
    counts = sorted(set(effects.loc[effects.family.isin(["induction", "previous", "both"]), "k"]))
    shown = counts
    if len(counts) > max_counts:          # a long ladder such as 1-32: the powers of two and the largest count
        shown = [k for k in counts if k & (k - 1) == 0 or k == counts[-1]][-max_counts:]
    keys = [(family, k) for family in SUMMARY_ORDER for k in sorted(set(effects.loc[effects.family == family, "k"]))
            if family.startswith("random") or k in shown]
    if not keys or not scopes:
        return None
    noun = {"induction": "induction", "previous": "previous-token", "both": "union"}

    def label(family, k):
        if family in noun:
            return f"{noun[family].capitalize()} · top {k}" + (" of each" if family == "both" else "")
        if family.endswith("_matched"):
            base = family[:-len("_matched")]
            return f"Layer-matched to {noun[base]} top {k}" + (" of each" if base == "both" else "")
        matched = effects.loc[(effects.family == family) & (effects.k == k), "matched_k"].iloc[0]
        return f"Random · {k} heads, as many as " + (f"both top {matched} together" if family == "random_union"
                                                      else f"top {matched}")

    values = np.full((len(keys), len(scopes)), np.nan)
    texts = [["" for _ in scopes] for _ in keys]
    for r, (family, k) in enumerate(keys):
        for c, scope in enumerate(scopes):
            match = effects[(effects.family == family) & (effects.k == k) & (effects.scope == scope)]
            if len(match):
                row = match.iloc[0]
                values[r, c] = 100 * row["drop"]
                interval = (f" [{_signed(row['drop_low'])}, {_signed(row['drop_high'])}]"
                            if np.isfinite(row["drop_low"]) else "")
                texts[r][c] = f"{100 * row['accuracy']:.1f}%\n{_signed(row['drop'])}{interval}"
    cross = None if crossover is None else crossover[crossover["k"].isin(shown)]
    n_cross = 0 if cross is None else len(cross)

    # Sizes in inches, so that the layout is the same for any number of rows.
    row_h, col_w, title_h, header_h, gap_h = 0.45, 2.15, 1.15, 0.45, 0.95
    foot_h = 0.45 if shown != counts else 0.2
    label_w = 0.35 + 0.072 * max(len(label(*key)) for key in keys)
    n_cols = max(len(scopes), 3)
    width = label_w + col_w * n_cols + 0.3
    height = title_h + header_h + row_h * len(keys) + (gap_h + header_h + row_h * n_cross if n_cross else 0) + foot_h
    fig = plt.figure(figsize=(width, height))

    def draw(top, grid, cell_texts, row_labels, column_labels, cmap):
        ax = fig.add_axes([label_w / width, 1 - (top + row_h * len(grid)) / height,
                           col_w * grid.shape[1] / width, row_h * len(grid) / height])
        limit = max(5.0, np.nanmax(np.abs(grid)) if np.isfinite(grid).any() else 0)   # pale below a few points
        colours = plt.get_cmap(cmap).copy()
        colours.set_bad("#f3f5f6")
        ax.imshow(np.ma.masked_invalid(grid), cmap=colours, vmin=-limit, vmax=limit, aspect="auto")
        for r, row_texts in enumerate(cell_texts):
            for c, text in enumerate(row_texts):
                if text:
                    dark = abs(grid[r, c]) > .55 * limit
                    ax.text(c, r, text, ha="center", va="center", fontsize=8.5, linespacing=1.3,
                            color="white" if dark else "#1d2b33")
        ax.set_xticks(range(grid.shape[1]), column_labels, fontsize=9.5)
        ax.xaxis.tick_top()
        ax.set_yticks(range(len(grid)), row_labels, fontsize=9)
        ax.tick_params(length=0)
        for spine in ax.spines.values():
            spine.set_visible(False)
        return ax

    ax = draw(title_h + header_h, values, texts, [label(*key) for key in keys],
              [SCOPE_LABEL[s] for s in scopes], "RdBu_r")
    for r in range(1, len(keys)):
        if keys[r][0] != keys[r - 1][0]:
            ax.axhline(r - .5, color="white", lw=4)
    intact = 100 * effects["intact"].iloc[0]
    fig.text(.5, 1 - .3 / height, f"Valid-neighbour probability over words {from_word:,}–{last_word:,}, "
             f"intact model {intact:.1f}%", ha="center", va="center", fontsize=12)
    fig.text(.5, 1 - .58 / height, context, ha="center", va="center", fontsize=9.5)
    fig.text(.5, 1 - .86 / height, "Each cell: mean probability, then the drop relative to the intact model in "
             "percentage points (positive: below the intact model) [95% bootstrap interval over walks]",
             ha="center", va="center", fontsize=8.5, color="#4d5b62")
    if n_cross:
        top = title_h + header_h + row_h * len(keys) + gap_h + header_h
        grid = 100 * cross[["previous_history_minus_final", "induction_history_minus_final", "crossover"]].to_numpy()
        cross_texts = [[_signed(row.previous_history_minus_final), _signed(row.induction_history_minus_final),
                        _signed(row.crossover) + (f"\n[{_signed(row.crossover_low)}, {_signed(row.crossover_high)}]"
                                                  if np.isfinite(row.crossover_low) else "")]
                       for row in cross.itertuples()]
        draw(top, grid, cross_texts, [f"Top {k}" for k in cross["k"]],
             ["Previous-token heads:\nhistory − final word", "Induction heads:\nhistory − final word",
              "Crossover:\nfirst − second"], "BrBG")
        fig.text(.5, 1 - (top - header_h - .42) / height,
                 "Crossover, percentage points: each head type's drop when removed in the history minus its drop when "
                 "removed at the final word.\nPositive: previous-token heads matter more in the history and induction "
                 "heads more at the final word.", ha="center", va="center", fontsize=8.5, color="#4d5b62",
                 linespacing=1.4)
    if shown != counts:
        fig.text(.5, .22 / height, f"Head counts shown: {', '.join(map(str, shown))}. Every count is in "
                 "effects-summary.csv and crossover-summary.csv.", ha="center", va="center", fontsize=8,
                 color="#4d5b62")
    path = folder / "effects-table.png"
    fig.savefig(path, dpi=140)
    plt.close(fig)
    return path


def render_results(cache_dir, output_dir, words, adjacency, positions, cutoffs, conditions,
                   model_name, graph_name, *, coordinates=None, expected_walks=None,
                   window_size=200, smooth=30, summary_from=None, seed=0):
    """Generate figures and portable aggregates. All arrays are CPU NumPy arrays."""
    cache_dir = Path(cache_dir)
    folder = Path(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    words = [str(w) for w in words]
    positions = np.asarray(positions, dtype=int)
    cutoffs = np.asarray(cutoffs, dtype=int)
    if len(positions) < 1 or positions[0] < 1 or np.any(np.diff(positions) <= 0):
        raise ValueError("Accuracy positions must be increasing, one-based word indices.")
    if len(cutoffs) < 1 or cutoffs[0] < 1 or np.any(np.diff(cutoffs) <= 0):
        raise ValueError("PCA cutoffs must be increasing, one-based context lengths.")
    if smooth < 1 or window_size < 1:
        raise ValueError("Window sizes must be positive.")
    adj = np.asarray(adjacency, dtype=bool)
    if adj.shape != (len(words), len(words)) or not np.array_equal(adj, adj.T):
        raise ValueError("Expected one undirected graph linking the supplied words.")
    edges = np.argwhere(np.triu(adj, k=1))
    if coordinates is None:
        angle = np.linspace(0, 2*np.pi, len(words), endpoint=False)
        coordinates = np.column_stack([np.cos(angle), np.sin(angle)])
    if isinstance(coordinates, dict):
        coordinates = [coordinates[i] for i in range(len(words))]
    graph_xy = np.asarray(coordinates, dtype=float)
    if graph_xy.shape != (len(words), 2) or not np.isfinite(graph_xy).all():
        raise ValueError("The reference graph needs one (x, y) pair for every node.")
    names = [c["name"] for c in conditions]
    if len(names) != len(set(names)) or sum(c["family"] == "intact" for c in conditions) != 1:
        raise ValueError("Give each condition a unique name and define one intact condition.")
    for c in conditions:
        if c["family"] != "intact" and (c["family"] not in FAMILY_LABEL or c["scope"] not in SCOPE_LABEL):
            raise ValueError(f"Unknown head family or scope: {c}.")
    mass_by_walk, xy, variance, node_count, n = read_summaries(
        cache_dir, words, positions, cutoffs, conditions, expected_walks, window_size)
    mass = mass_by_walk.mean(axis=1)
    overlap = spectral_alignment(adj, xy)
    if summary_from is None:
        summary_from = int(positions[-1]) // 2 + 1
    effects, crossover = summarize_effects(mass_by_walk, positions, conditions, summary_from, seed=seed)
    effects.to_csv(folder / "effects-summary.csv", index=False)
    if crossover is not None:
        crossover.to_csv(folder / "crossover-summary.csv", index=False)

    compact = folder / "aggregated-results.npz"
    np.savez_compressed(compact, accuracy=mass.astype(np.float32), accuracy_by_walk=mass_by_walk.astype(np.float32),
                        positions=positions, words=words,
                        condition_names=names, pca=xy.astype(np.float32), variance_fraction=variance,
                        graph_overlap=overlap,
                        cutoffs=cutoffs, window_size=window_size, node_count=node_count, walks=n,
                        edges=edges, graph_coordinates=graph_xy)
    with (folder/"accuracy.csv").open("w", newline="", encoding="utf-8") as file:
        rows = csv.writer(file)
        rows.writerow(["context_words", *names])
        for i, pos in enumerate(positions):
            rows.writerow([pos, *mass[:, i].astype(np.float32)])
    with (folder/"node-map.csv").open("w", newline="", encoding="utf-8") as file:
        rows = csv.writer(file)
        rows.writerow(["node", "word", "reference_x", "reference_y"])
        for i, word in enumerate(words):
            rows.writerow([i, word, *graph_xy[i]])
    with (folder/"results-schema.json").open("w", encoding="utf-8") as file:
        json.dump({"model": model_name, "graph": graph_name, "walks": n, "conditions": conditions,
                   "accuracy": "Mean full-vocabulary probability assigned to legal neighbors; units 0–1.",
                   "accuracy_by_walk": "Axes [condition, walk, position]: the same probability for every walk.",
                   "effects_summary": f"effects-summary.csv: mean over words {summary_from}–{int(positions[-1])}, drop "
                                      "relative to intact on the same walks, 95% paired bootstrap interval over walks, "
                                      "and the drop beyond the matching control where one exists.",
                   "crossover_summary": "crossover-summary.csv: per k, (previous-token drop in the history minus at the "
                                        "final word) minus (the same for induction heads), with a 95% interval.",
                   "pca": "Axes [condition, cutoff, node, component]. NaN coordinates mark unobserved nodes.",
                   "node_count": "Pooled number of observations per cutoff and node in the trailing PCA window.",
                   "variance_fraction": "Fraction of between-word centroid variance on PC1 and PC2.",
                   "graph_overlap": "Mean squared alignment with the first two nonconstant modes of the unnormalized graph Laplacian (0–1). NaN for incomplete coverage, rank below two, fewer than three nodes, disconnected graphs, or a tie crossing the second/third nonconstant eigenvalue boundary. With exactly three nodes, any full-rank centered plane spans the entire nonconstant subspace.",
                   "random_curves": "Random draws of the same size are averaged for accuracy and plotted separately for PCA."},
                  file, indent=2, ensure_ascii=False)

    readme = f"""Results of the graph-walk head-ablation notebook: {model_name}, {graph_name}, {len(words)} words, {n} walks.

accuracy.csv             mean valid-neighbour probability (0-1) at every scored context length, one column per
                         condition (unsmoothed).
effects-summary.csv      per head group, head count k and scope: mean valid-neighbour probability over words
                         {summary_from}-{int(positions[-1])}, its drop relative to the intact model on the same walks
                         with a 95% paired bootstrap interval over walks and, where a matching control exists, the
                         control's drop and the drop beyond the control with its interval. Rows: ablated groups,
                         layer-matched controls, then random controls. "random" has as many heads as the largest
                         single-family group, "random_union" as many as the largest union; for these rows k is the
                         number of random heads and matched_k the largest head count.
crossover-summary.csv    per k: (previous-token drop in the history - at the final word) minus (the same for
                         induction heads), with a 95% bootstrap interval over walks.
effects-table.png        both tables as one image, coloured by the size of each drop.
aggregated-results.npz   numpy.load(...): accuracy [condition, position], accuracy_by_walk [condition, walk, position],
                         positions, condition_names, pca [condition, cutoff, word, 2] (NaN = word not yet seen),
                         variance_fraction, graph_overlap [condition, cutoff], cutoffs, window_size,
                         node_count [cutoff, word], walks, edges, graph_coordinates, words.
node-map.csv             the word of every vertex and its position in the reference drawing.
results-schema.json      the list of conditions and a description of every array.
experiment.json          model revision, graph, words and token ids, head rankings and groups, seeds and settings.
execution-settings.json  GPU and chunk sizes of each session that contributed to the run.
native-check.json        differences between the engine and ordinary Hugging Face forward passes.
*.png, *.pdf             the figures.
"""
    (folder / "README.txt").write_text(readme, encoding="utf-8")

    noun = "walk" if n == 1 else "walks"
    context = f"{model_name} · {graph_name} · {len(words)} nodes · {n} {noun}"
    accuracy_pdf, accuracy_png, scope_png = _accuracy_figures(folder, positions, mass_by_walk, conditions, context,
                                                              smooth)
    effects_png = _effects_figure(folder, effects, context, summary_from, int(positions[-1]))
    table_png = _effects_table_figure(folder, effects, crossover, context, summary_from, int(positions[-1]))
    pca_pdf, pca_png = _pca_figures(folder, words, graph_xy, edges, xy, variance, overlap, node_count,
                                     conditions, cutoffs, window_size, context)
    return {"accuracy_png": accuracy_png, "scope_png": scope_png, "effects_png": effects_png, "table_png": table_png,
            "pca_png": pca_png,
            "accuracy_pdf": accuracy_pdf, "pca_pdf": pca_pdf, "summary_npz": compact,
            "effects_table": effects, "crossover_table": crossover}

In [ ]:
#@title Plot, summarise, package and download the results
from IPython.display import Image, display

files = render_results(cache_dir, export_dir, words, adjacency, score_at, cutoffs, conditions,
                       MODEL_ID, graph_name, coordinates=graph_xy, expected_walks=NUMBER_OF_SEQUENCES,
                       window_size=PCA_WINDOW, smooth=30, summary_from=summary_from, seed=SEED)
for key in ("accuracy_png", "scope_png", "effects_png", "table_png", "pca_png"):
    if files[key] is not None:
        display(Image(filename=str(files[key]), width=1000))
columns = ["family", "k", "scope", "accuracy", "drop", "drop_low", "drop_high"]
columns += [c for c in ("control", "control_drop", "drop_beyond_control", "beyond_low", "beyond_high")
            if c in files["effects_table"]]
display(files["effects_table"][columns].round(4))
if files["crossover_table"] is not None:
    display(files["crossover_table"].round(4))

# Only the public results directory enters the ZIP. The resumable working cache stays on disk.
archive = Path(shutil.make_archive(str(run_dir / f"graph-results-{fingerprint}"), "zip", root_dir=export_dir))
print(f"{archive} ({archive.stat().st_size / 2**20:.1f} MiB)")
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(archive))
    except ImportError:
        print("Not running in Colab: the ZIP is in the run folder above.")

## How to read the results
- **Start with the intact curve.** If the intact model's valid-neighbour probability has not clearly risen by the end of
  the walk, every ablation effect is small for a trivial reason; use longer walks.
- **Scope pattern.** The induction-circuit prediction is a large drop for previous-token heads removed in the history and
  for induction heads removed at the final word, and smaller drops in the other two cells. The crossover statistic sums
  this up in one number per k.
- **Controls.** Compare every drop with its control at the same scope. The random control has the size of the largest
  group, so it is a fair comparison only for the largest k; the layer-matched control fits every k.
- **What the maps can show.** If previous-token heads build the geometry by mixing each word with its predecessor at the
  word itself (Arditi's account), the map should lose its graph shape when they are switched off at the final word,
  which here means at every word, and largely keep it when they are switched off only in the history.
- **History includes the previous word.** "History only" switches heads off at every earlier position, including the word
  just before the prediction, so it cannot separate the processing of that word from information built up earlier in the
  walk.
- **Head rankings come from a synthetic probe.** A head's behaviour on repeated random tokens need not be its only role on
  graph walks, and the two top-k lists can overlap; the union condition removes shared heads once.
- **One graph and one word assignment per run.** Repeat with other `SEED` and `WORD_ASSIGNMENT_SEED` values to see how
  much the results depend on the walks and on the words.

## Technical details and reproducibility

**Software.** The notebook is written for Google Colab's GPU runtime (Python 3). Its installation cell pins
`transformers==4.56.2` and `accelerate==1.10.1`, and requires `networkx` ≥ 3.2 (< 4), `matplotlib` ≥ 3.6,
`pandas` ≥ 1.5, `sentencepiece` ≥ 0.1.99, `filelock` ≥ 3 and `tqdm` ≥ 4. If it has to replace a Transformers version
that is already loaded, it asks for one restart of the session. PyTorch and NumPy are Colab's preinstalled versions;
the cell prints the Transformers and PyTorch versions. The engine (section 3) re-implements the
Llama, Qwen2 and Mistral blocks of Transformers 4.56.2, and the engine check (section 4) compares it with that
version's own forward pass before every run. Other architectures are not supported.

**Model revisions.** Llama 3.1 8B is pinned to revision `d04e592bb4f6aa9cfee91e2e20afa771667e1d4b`, the revision for
which the stored ranking below is used. Every other model is loaded at its latest revision (`main`), and the commit
actually loaded is saved as `model_revision` in `experiment.json`. The stored Llama 3.2 1B ranking is used only if the
SHA-256 of the checkpoint's `model.safetensors` is
`68a2e4be76fa709455a60272fba8e512c02d81c46e6c671cc9449e374fd6809a`.

**Stored ranking of Llama 3.1 8B.** It comes from `head_scores.npz` in Arditi's repository (linked in the references;
SHA-256 of the file: `78edcdfdef92fb994cb7f0e4141a0cecf344434169325ac1ce0c22b167cfe174`), whose arrays
`induction_scores` and `prev_token_scores` hold one score per block and query head (32 × 32). The stored lists are
the 32 highest scores of each array, in order of score; this was checked against the file. There are no ties at the
cut: the 32nd and 33rd scores are 0.1941 and 0.1901 for induction heads, and 0.2130 and 0.2115 for previous-token
heads. The two lists share no head, so the union of the top 32 of each removes 64 heads, as does its random control.

**Precision.** With `PRECISION = "auto"`, models run in bfloat16 on GPUs of compute capability 8 or higher (L4,
A100), and in float16 on older GPUs (T4), except Qwen models, which then run in float32. The head probe always
recomputes attention probabilities in float32. The engine check accepts differences up to 2×10⁻⁴ in float32 and 0.04
in 16-bit precision. Runs on different GPUs or library versions agree up to such floating-point differences; the
graph, words, walks and head groups do not depend on the hardware.

**Random numbers.** `SEED` fixes the random graph (for `TOPOLOGY = "random"`) and the walks; each walk has its own
random stream, so adding walks leaves the earlier ones unchanged. `WORD_ASSIGNMENT_SEED` fixes which words go on which
vertex. The head probe uses seed 37139. The random-head controls use seed 113295: the single-family draws use it
directly, the union-size draws `[113295, 2]` and the layer-matched draws `[113295, 1]`, and every draw has its own
child seed, so adding draws keeps the earlier ones. The bootstrap of the summary table uses `SEED`.

**What every run records.** `experiment.json` holds the model and its revision, the precision, the Transformers and
PyTorch versions, the GPU, a fingerprint of the measuring code, the seeds, the graph, the words and their token ids,
the head rankings and their origin, the head groups and the conditions. `execution-settings.json` lists the GPU and
chunk sizes of each session that contributed, and `native-check.json` the differences found by the engine check. The
run folder's name ends with a fingerprint of all these settings, so results from different code, models or settings
are never combined.

**Tests of the code.** The notebook was run end to end on CPU with small randomly initialised Llama, Qwen2 (with
attention biases) and Mistral models in float32 (PyTorch 2.8.0, Transformers 4.56.2). The engine matched Hugging Face
forward passes with hooks to about 10⁻⁷ in all four conditions, including the blocked attention path; two deliberately
introduced engine errors were caught by the engine check; all topologies ran, with and without each kind of control,
and an interrupted run resumed correctly. 16-bit precision is checked only by the engine check of each run.

**Runs reported in the write-up.** They were made on Google Colab in November 2026: Llama-3.1-8B and other 7–8B
models on an NVIDIA L4 in bfloat16, and 1–1.5B models on a T4 (float16 for Llama and float32 for Qwen, as chosen by
`PRECISION = "auto"`). The Llama-3.1-8B runs used PyTorch 2.14.1 (CUDA 13.0) and Transformers 4.56.2; every run's
`experiment.json` records its exact versions, GPU and model revision.

### References

[Park et al., *ICLR: In-Context Learning of Representations*](https://arxiv.org/abs/2501.00070) · [Arditi, ICLR Blogposts 2026](https://iclr-blogposts.github.io/2026/blog/2026/iclr-induction/) · [Published 8B code and scores](https://github.com/andyrdt/iclr_induction/tree/908ddb0b02a3184b59a5dba6bc068e90a1d30400) · [Meta Llama](https://huggingface.co/meta-llama/Llama-3.1-8B) · [Qwen2.5 base models](https://huggingface.co/Qwen/Qwen2.5-1.5B) · [Mistral 7B v0.3](https://huggingface.co/mistralai/Mistral-7B-v0.3)

Software: [PyTorch](https://pytorch.org) · [Hugging Face Transformers](https://github.com/huggingface/transformers) · [NetworkX](https://networkx.org) · [NumPy](https://numpy.org) · [pandas](https://pandas.pydata.org) · [Matplotlib](https://matplotlib.org). Each model is subject to the licence on its Hugging Face page; Llama models are gated.